# Ch 30 · Capstone: GPU-Native Reinforcement Learning at Scale

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozzyozbourne/ai_lab/blob/master/dpdk-research/mujoco-warp-course/notebooks/30-capstone-rl-at-scale.ipynb)

Companion notebook for the chapter page `chapters/30-capstone-rl-at-scale.html` ([page](../chapters/30-capstone-rl-at-scale.html)). Pinned versions: mujoco-warp 4.0.1, warp-lang 1.19.0, mujoco-core 4.0.1.

**Before you start:** Runtime > Change runtime type > select a GPU. Then run the cells top to bottom.

In [ ]:
# Check the GPU Colab gave you (this fails harmlessly on a CPU runtime).
!nvidia-smi

In [ ]:
# Standard setup cell (identical in every notebook). Run it first.
# 1. Rendering backend must be chosen BEFORE mujoco is imported (rendering notebooks only).
import os
os.environ.setdefault("MUJOCO_GL", "egl")
# 2. Driver check: pip warp-lang 1.19.0 is built with CUDA 13 and needs an R580+ driver.
import shutil, subprocess
smi = ""
if shutil.which("nvidia-smi"):
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip()
    except OSError:
        smi = ""
print(smi or "No NVIDIA GPU visible")
try:
    major = int(smi.split(".")[0]) if smi else 0
except ValueError:
    major = 0
# 3. Pick the Warp build: PyPI (CUDA 13) for R580+, else the +cu12 release wheel.
WARP = ("warp-lang==1.19.0" if major >= 580 else
        "https://github.com/NVIDIA/warp/releases/download/v1.19.0/"
        "warp_lang-1.19.0+cu12-py3-none-manylinux_2_28_x86_64.whl")
# 4. Install pinned versions ([viewer] pulls glfw + pyopengl + pillow via mujoco-core[rendering]).
!pip install -q "mujoco-warp[viewer]==4.0.1" "mujoco-core[rendering]==4.0.1" {WARP} mediapy matplotlib
# 5. Import, quiet the JIT log, verify that Warp itself sees the GPU (otherwise it silently uses CPU).
import importlib.metadata as md
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco
import mujoco_warp as mjw
wp.init()
ON_GPU = wp.get_cuda_device_count() > 0
if not ON_GPU:
    print("WARNING: Warp sees no GPU. Runtime > Change runtime type > GPU. Using small CPU budgets.")
print("device:", wp.get_device(), "| warp", wp.__version__, "| mujoco-core", md.version("mujoco-core"),
      "| mujoco-warp", md.version("mujoco-warp"))
GRAPH_COND = ON_GPU and wp.is_conditional_graph_supported()
print("conditional graphs supported:", GRAPH_COND)

This notebook is the companion of Chapter 30. Run the setup cell first (GPU runtime). The next cells write the helper files the examples import: `my_env.py` (the running project, snapshot after Chapter 30), `ex1_env.py` (CartpoleEnv), `ppo_reference.py` (the reference PPO) and `ex10_humanoid_env.py` (HumanoidStandEnv). Every number printed below is your own; the page only quotes CPU numbers from the course's verification box.

**Budgets** (targets, not measurements: the course was verified on a CPU). Warm-up track (Examples 30.1-30.8): sized for about 10-20 minutes on a Colab GPU, most of it Example 30.6. Contact track (Examples 30.9-30.11): its own 20-30 minutes, most of it Example 30.11. Stretch sections (30.12, 30.13) are optional.

In [ ]:
%%writefile my_env.py
"""my_env.py - the course's running helper module, snapshot after Chapter 30.

Ch 6 added make_worlds. Ch 8 added reset_done, nonfinite_reset, snapshot and restore.
Ch 9 added describe_overflow (Example 9.7).
Ch 10 added randomize (Example 10.9): in-place, graph-friendly friction re-randomization.
Ch 14 added sensor_slices (Example 14.3): sensor name -> (adr, dim) in d.sensordata.
Ch 24 added needs_conditional_fallback and capture_step (Example 24.4): record nsub steps (plus
optional user kernels) into one CUDA graph, with the graph-conditional fallback for CUDA < 12.4.
Ch 30 adds scale_body_mass (eager mass domain randomization + set_const, Example 30.7). The two
RL environments built on top of this module live in ex1_env.py (CartpoleEnv) and
ex10_humanoid_env.py (HumanoidStandEnv).
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

__all__ = ["make_worlds", "reset_done", "nonfinite_reset", "snapshot", "restore",
           "describe_overflow", "randomize", "sensor_slices", "needs_conditional_fallback",
           "capture_step", "MASS_DERIVED", "scale_body_mass"]


# --- Ch 6 ----------------------------------------------------------------------------------------
def make_worlds(xml, nworld, nconmax=None, njmax=None, batch_sizes=None, **sizes):
    """Compile `xml` and return (mjm, m, d) with `nworld` worlds (Ch 6 signature).

    nconmax / njmax are per world; batch_sizes (e.g. {"body_mass": nworld}) gives chosen Model
    fields their own rows (Ch 10); any other keyword (naconmax=, ...) goes to make_data.
    """
    mjm = mujoco.MjModel.from_xml_string(xml)
    m = mjw.put_model(mjm, batch_sizes=batch_sizes)
    d = mjw.make_data(mjm, nworld=nworld, nconmax=nconmax, njmax=njmax, **sizes)
    assert d.qpos.shape == (nworld, mjm.nq), d.qpos.shape
    return mjm, m, d


# --- Ch 8: resets --------------------------------------------------------------------------------
@wp.kernel
def _nonfinite_kernel(qpos_in: wp.array2d[float], qvel_in: wp.array2d[float],
                      done_out: wp.array[bool]):
    worldid = wp.tid()
    bad = bool(False)
    for i in range(qpos_in.shape[1]):
        if not wp.isfinite(qpos_in[worldid, i]):
            bad = True
    for i in range(qvel_in.shape[1]):
        if not wp.isfinite(qvel_in[worldid, i]):
            bad = True
    if bad:
        done_out[worldid] = True          # only ever sets flags; never clears another reason


def reset_done(m, d, done):
    """Reset the worlds whose done flag is True, then recompute derived quantities.

    `done` is a wp.array of bool with shape (nworld,). Everything stays on the device, so this
    can be captured in a CUDA graph together with mjw.step (Ch 24, Ch 26).
    """
    mjw.reset_data(m, d, reset=done)
    mjw.forward(m, d)                     # reset_data does not recompute xpos, sensors, ...


def nonfinite_reset(m, d, done):
    """OR `done` with "qpos or qvel has a NaN/Inf" (MJWarp has no mj_checkPos/mj_checkVel)."""
    wp.launch(_nonfinite_kernel, dim=d.nworld, inputs=[d.qpos, d.qvel], outputs=[done])


# --- Ch 8: state snapshots -----------------------------------------------------------------------
def snapshot(m, d, mjm, sig, buf=None):
    """Copy the state components in `sig` of every world into a (nworld, size) float array.

    Use State.PHYSICS for MPC (it includes HISTORY and PLASTIC; note it has no TIME bit),
    add State.WARMSTART for bitwise replays with contacts, State.INTEGRATION for everything.
    """
    size = mujoco.mj_stateSize(mjm, int(sig))
    if buf is None:
        buf = wp.zeros((d.nworld, size), dtype=float)
    # get_state does NOT check the size: an undersized buffer is written out of bounds.
    assert buf.shape == (d.nworld, size), f"state buffer {buf.shape} != {(d.nworld, size)}"
    mjw.get_state(m, d, buf, int(sig))
    return buf


def restore(m, d, mjm, buf, sig, active=None):
    """Write a snapshot back (only where the per-world bool mask `active` is True, if given)."""
    assert buf.shape == (d.nworld, mujoco.mj_stateSize(mjm, int(sig)))
    mjw.set_state(m, d, buf, int(sig), active)


# --- Ch 9: overflow ------------------------------------------------------------------------------
def describe_overflow(d, max_worlds=8):
    """Decode d.overflow: "none", or one line per flagged world plus a world count per bit.

    Loops over mjw.OverflowType (Python >= 3.11 iterates the single-bit members only), so bits
    added by a later MJWarp release are decoded without editing this function.
    """
    ov = d.overflow.numpy()
    if not ov.any():
        return "none"
    lines = []
    flagged = np.nonzero(ov)[0]
    for w in flagged[:max_worlds]:
        names = [bit.name for bit in mjw.OverflowType if ov[w] & bit]
        lines.append(f"world {w}: " + " | ".join(names))
    if len(flagged) > max_worlds:
        lines.append(f"... and {len(flagged) - max_worlds} more flagged worlds")
    counts = [f"{bit.name} x{int(np.count_nonzero(ov & bit))}" for bit in mjw.OverflowType
              if (ov & bit).any()]
    lines.append("worlds per bit: " + ", ".join(counts))
    return "\n".join(lines)


# --- Ch 10: domain randomization ----------------------------------------------------------------
@wp.kernel
def _randomize_friction_kernel(
    seed: wp.array(dtype=int),          # device array: a graph replays with the CURRENT value
    episode: wp.array(dtype=int),       # per-world episode counter, bumped on every re-draw
    reset: wp.array(dtype=bool),        # which worlds to re-draw (all if every entry is True)
    lo: float,
    hi: float,
    geom_friction: wp.array2d(dtype=wp.vec3),   # m.geom_friction, shape (nworld, ngeom)
):
    worldid, geomid = wp.tid()
    if not reset[worldid]:
        return
    # one random stream per (seed, world, episode): reproducible and different every episode
    rng = wp.rand_init(seed[0], worldid * 7919 + episode[worldid] * 104729)
    mu = wp.randf(rng, lo, hi)
    f = geom_friction[worldid, geomid]
    geom_friction[worldid, geomid] = wp.vec3(mu, f[1], f[2])


@wp.kernel
def _bump_episode_kernel(reset: wp.array(dtype=bool), episode: wp.array(dtype=int)):
    worldid = wp.tid()
    if reset[worldid]:
        episode[worldid] = episode[worldid] + 1


def randomize(m, seed, episode, reset, lo=0.1, hi=1.0):
    """Draw a new sliding friction in [lo, hi) for every geom of the worlds marked in reset.

    Writes m.geom_friction in place (it must already have one row per world, e.g. from
    put_model(batch_sizes={"geom_friction": nworld})), so a captured CUDA graph that contains
    this call keeps working: only array CONTENTS change, never array objects.
    seed: wp.array(dtype=int, shape=(1,)); episode: wp.array(dtype=int, shape=(nworld,));
    reset: wp.array(dtype=bool, shape=(nworld,)).
    """
    nworld = reset.shape[0]
    assert m.geom_friction.shape[0] == nworld, "batch geom_friction to nworld first"
    wp.launch(_randomize_friction_kernel, dim=(nworld, m.ngeom),
              inputs=[seed, episode, reset, lo, hi], outputs=[m.geom_friction])
    wp.launch(_bump_episode_kernel, dim=nworld, inputs=[reset], outputs=[episode])


# --- Ch 14: sensors ------------------------------------------------------------------------------
def sensor_slices(mjm):
    """Return {sensor name: (adr, dim)} so d.sensordata[:, adr:adr + dim] is that sensor.

    Unnamed sensors get the key "sensor<i>". The addresses come from the compiled MjModel
    (mjm.sensor_adr, mjm.sensor_dim) and are the same in every world.
    """
    out = {}
    for i in range(mjm.nsensor):
        name = mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_SENSOR, i) or f"sensor{i}"
        out[name] = (int(mjm.sensor_adr[i]), int(mjm.sensor_dim[i]))
    return out


# --- Ch 24: CUDA graphs ---------------------------------------------------------------------------
def needs_conditional_fallback(device=None):
    """True when `device` is a CUDA device whose toolkit or driver is older than CUDA 12.4.

    There, CUDA graph conditional nodes (wp.capture_while / wp.capture_if) do not exist, and
    put_model only *warns* (warp_util.check_toolkit_driver) while m.opt.graph_conditional stays True.
    On the CPU device the answer is False: Warp 1.19 records CPU graphs with APIC, which replays
    capture_while / capture_if callbacks without CUDA.
    """
    device = wp.get_device(device)
    return device.is_cuda and not wp.is_conditional_graph_supported()


def capture_step(m, d, nsub=1, post=None, verbose=True):
    """Record `nsub` calls of mjw.step(m, d), then `post(m, d)` if given, into one graph.

    Returns the graph; replay it with wp.capture_launch(graph). Capturing records the launches,
    it does not run them, so d is not advanced. The graph holds the *addresses* of every array it
    touches and the Python decisions taken while recording (integrator, solver, iteration
    counts, flags): change array contents in place (assign, fill_, copy, kernels) and recapture
    after replacing an array or changing a Python-level option (the Ch 24 knob board).
    """
    assert nsub >= 1, "nsub must be >= 1"
    device = d.qpos.device
    if needs_conditional_fallback(device) and m.opt.graph_conditional:
        m.opt.graph_conditional = False     # fixed iteration counts instead of device-side loops
        if verbose:
            print("capture_step: CUDA < 12.4, set m.opt.graph_conditional = False")
    with wp.ScopedCapture(device=device) as capture:
        for _ in range(nsub):
            mjw.step(m, d)
        if post is not None:
            post(m, d)
    return capture.graph


# --- Ch 30: eager mass domain randomization -------------------------------------------------------
# derived fields that set_const recomputes from body_mass/body_inertia (Ch 10 ex5/ex6)
MASS_DERIVED = ["body_subtreemass", "body_invweight0", "dof_invweight0", "actuator_acc0"]


def scale_body_mass(mjm, m, d, body, scale):
    """Set body_mass[w, body] = scale[w] * nominal mass in every world, then run mjw.set_const.

    body_mass, body_inertia AND the derived fields in MASS_DERIVED must be batched to nworld
    (put_model(batch_sizes={...})): set_const launches over each OUTPUT's leading size, so an
    unbatched body_subtreemass would get world 0's value for every world (Ch 10 ex6).
    The inertia is scaled by the same factor, as the set_const docstring recommends. set_const
    recomputes the derived fields (body_subtreemass, dof_invweight0, actuator_acc0, ...); it
    allocates temporaries and relaunches every world, so call it EAGERLY between rollouts, never
    inside a captured graph (Ch 10 §6-§7, Ch 30 §9).
    """
    scale = np.asarray(scale, dtype=np.float32).reshape(-1)
    assert m.body_mass.shape[0] == d.nworld == scale.shape[0], "batch body_mass to nworld first"
    for name in MASS_DERIVED:
        assert getattr(m, name).shape[0] == d.nworld, f"batch {name} to nworld too"
    mass = np.tile(mjm.body_mass.astype(np.float32), (d.nworld, 1))
    inertia = np.tile(mjm.body_inertia.astype(np.float32), (d.nworld, 1, 1))
    mass[:, body] *= scale
    inertia[:, body] *= scale[:, None]
    m.body_mass.assign(mass)                      # same array objects: only contents change
    m.body_inertia.assign(inertia)
    mjw.set_const(m, d)                           # SetConstGroup.ALL (body_mass needs ALL)

In [ ]:
%%writefile ex1_env.py
"""Example 30.1 - CartpoleEnv: a fully on-device RL environment for thousands of worlds.

One call of env.step(action) replays ONE captured graph that does everything on the device:
  1. copy the policy's action into d.ctrl (clamped),
  2. n_substeps physics steps (mjw.step),
  3. reward / termination / truncation kernel (with a non-finite guard),
  4. masked auto-reset of the finished worlds: mjw.reset_data, a random initial state, and an
     in-graph domain-randomization kernel (motor gain, slider damping) for the NEW episode,
  5. the observation kernel and the episode bookkeeping (returns and lengths).
The host never reads a value inside the loop. backend="torch" hands out zero-copy torch tensors
(Ch 28); backend="warp" hands out the wp.arrays themselves, so the env also works without torch.
Other examples import this file: `from ex1_env import CartpoleEnv`.
Run: python ex1_env.py
"""
import pathlib
import sys

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

HERE = pathlib.Path(__file__).parent if "__file__" in globals() else pathlib.Path.cwd()
sys.path.insert(0, str(HERE))
try:
    import my_env                           # notebook: written by the %%writefile cell
except ImportError:
    import my_env_step30 as my_env          # examples/30-capstone-rl-at-scale/

CARTPOLE = """
<mujoco model="cartpole">
  <option timestep="0.01" integrator="implicitfast"/>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="front" pos="0 -2 2.1" xyaxes="1 0 0 0 1 2"/>
    <geom name="floor" type="plane" size="20 20 .1" rgba=".75 .8 .85 1" contype="0" conaffinity="0"/>
    <geom name="rail" type="capsule" fromto="-2 0 1 2 0 1" size=".02" contype="0" conaffinity="0"/>
    <body name="cart" pos="0 0 1">
      <joint name="slider" type="slide" axis="1 0 0" range="-1.8 1.8" limited="true" damping="0.1"/>
      <geom name="cart" type="box" size=".2 .1 .05" mass="1" rgba=".15 .3 .7 1" contype="0" conaffinity="0"/>
      <body name="pole">
        <joint name="hinge" type="hinge" axis="0 1 0" damping="0.01"/>
        <geom name="pole" type="capsule" fromto="0 0 0 0 0 .6" size=".03" mass="0.1" rgba=".9 .45 .1 1"
              contype="0" conaffinity="0"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="push" joint="slider" gear="10" ctrlrange="-1 1" ctrllimited="true"/></actuator>
  <sensor>
    <jointpos joint="slider"/><jointpos joint="hinge"/>
    <jointvel joint="slider"/><jointvel joint="hinge"/>
  </sensor>
</mujoco>
"""

vec10 = wp.types.vector(length=10, dtype=float)    # the dtype of m.actuator_gainprm
OBS_DIM = 5          # x, cos(theta), sin(theta), x_dot, theta_dot
ACT_DIM = 1


@wp.kernel
def _apply_action(action: wp.array2d(dtype=float), ctrl: wp.array2d(dtype=float)):
    worldid, i = wp.tid()
    ctrl[worldid, i] = wp.clamp(action[worldid, i], -1.0, 1.0)


@wp.kernel
def _reward_done(
    qpos: wp.array2d(dtype=float), qvel: wp.array2d(dtype=float),
    ctrl: wp.array2d(dtype=float),
    swingup: int, max_steps: int, x_limit: float, cos_limit: float,
    t: wp.array(dtype=int),
    reward: wp.array(dtype=float), terminated: wp.array(dtype=bool),
    truncated: wp.array(dtype=bool), done: wp.array(dtype=bool),
    final_obs: wp.array2d(dtype=float),
):
    worldid = wp.tid()
    x = qpos[worldid, 0]
    th = qpos[worldid, 1]
    xd = qvel[worldid, 0]
    thd = qvel[worldid, 1]
    a = ctrl[worldid, 0]
    steps = t[worldid] + 1
    t[worldid] = steps
    # non-finite guard: MJWarp has no mj_checkPos/mj_checkVel, so the env checks itself (Ch 8)
    bad = not (wp.isfinite(x) and wp.isfinite(th) and wp.isfinite(xd) and wp.isfinite(thd))
    c = wp.cos(th)
    term = bad or wp.abs(x) > x_limit
    if swingup == 0:
        term = term or c < cos_limit              # balance task: the pole fell
    trunc = (not term) and steps >= max_steps     # time limit: NOT a failure
    r = float(0.0)
    if not bad:
        if swingup == 0:
            r = c - 0.25 * x * x - 0.01 * a * a     # upright, centred, gentle
        else:
            # energy shaping: pole energy (normalized so upright at rest = 1) should reach 1.
            # Kept >= about -0.15 so that crashing into the rail end never pays off.
            e = (0.5 * 0.012 * thd * thd + 0.2943 * c) / 0.2943
            r = 0.5 * (1.0 + c) + 0.5 * wp.exp(-2.0 * wp.abs(e - 1.0))
            r = r - 0.05 * x * x - 0.01 * a * a
    reward[worldid] = r
    terminated[worldid] = term
    truncated[worldid] = trunc
    done[worldid] = term or trunc
    # the observation of the state the episode ENDED in (needed to bootstrap truncations)
    final_obs[worldid, 0] = x
    final_obs[worldid, 1] = c
    final_obs[worldid, 2] = wp.sin(th)
    final_obs[worldid, 3] = xd
    final_obs[worldid, 4] = thd
    if bad:
        for i in range(5):
            final_obs[worldid, i] = 0.0


@wp.kernel
def _init_and_randomize(
    seed: wp.array(dtype=int), episode: wp.array(dtype=int), reset: wp.array(dtype=bool),
    swingup: int, gain_lo: float, gain_hi: float, damp_lo: float, damp_hi: float,
    qpos: wp.array2d(dtype=float), qvel: wp.array2d(dtype=float),
    actuator_gainprm: wp.array2d(dtype=vec10),
    dof_damping: wp.array2d(dtype=float),
):
    worldid = wp.tid()
    if not reset[worldid]:
        return
    # one random stream per (seed, world, episode): reproducible, new for every episode
    rng = wp.rand_init(seed[0], worldid * 7919 + episode[worldid] * 104729)
    # random initial state (reset_data put qpos0 = upright, qvel = 0)
    qpos[worldid, 0] = wp.randf(rng, -0.2, 0.2)
    th0 = wp.randf(rng, -0.1, 0.1)
    if swingup != 0:
        th0 = th0 + wp.pi
    qpos[worldid, 1] = th0
    qvel[worldid, 0] = wp.randf(rng, -0.1, 0.1)
    qvel[worldid, 1] = wp.randf(rng, -0.1, 0.1)
    # in-graph domain randomization: fields with NO derived quantities (no set_const needed)
    g = actuator_gainprm[worldid, 0]
    g[0] = wp.randf(rng, gain_lo, gain_hi)        # motor gain (force = gain * gear * ctrl)
    actuator_gainprm[worldid, 0] = g
    dof_damping[worldid, 0] = wp.randf(rng, damp_lo, damp_hi)     # slider damping
    episode[worldid] = episode[worldid] + 1


@wp.kernel
def _observe(
    qpos: wp.array2d(dtype=float), qvel: wp.array2d(dtype=float),
    done: wp.array(dtype=bool), reward: wp.array(dtype=float), t: wp.array(dtype=int),
    obs: wp.array2d(dtype=float),
    ep_return: wp.array(dtype=float), last_return: wp.array(dtype=float),
    last_length: wp.array(dtype=int), n_finished: wp.array(dtype=int),
):
    worldid = wp.tid()
    th = qpos[worldid, 1]
    obs[worldid, 0] = qpos[worldid, 0]
    obs[worldid, 1] = wp.cos(th)
    obs[worldid, 2] = wp.sin(th)
    obs[worldid, 3] = qvel[worldid, 0]
    obs[worldid, 4] = qvel[worldid, 1]
    ret = ep_return[worldid] + reward[worldid]
    if done[worldid]:                    # bookkeeping on the device: no host sync per step
        last_return[worldid] = ret
        last_length[worldid] = t[worldid]
        n_finished[worldid] = n_finished[worldid] + 1
        ret = 0.0
        t[worldid] = 0
    ep_return[worldid] = ret


class CartpoleEnv:
    """nworld cartpoles stepped as one batch. step() is one graph replay (or eager calls).

    obs (nworld, 5), reward (nworld,), terminated / truncated (nworld,) bool, final_obs (nworld, 5).
    Finished worlds are reset INSIDE step(): the returned obs of a finished world is already the
    first observation of its next episode, and final_obs holds the observation it ended in.
    """

    def __init__(self, nworld, backend="torch", n_substeps=2, max_steps=200, seed=0,
                 swingup=False, gain_range=(0.8, 1.2), damping_range=(0.05, 0.2),
                 use_graph=True):
        self.nworld, self.n_substeps, self.max_steps = nworld, n_substeps, max_steps
        self.swingup = int(swingup)
        self.gain_range, self.damping_range = gain_range, damping_range
        self.mjm = mujoco.MjModel.from_xml_string(CARTPOLE)
        self.dt = n_substeps * float(self.mjm.opt.timestep)          # control period [s]
        # one row per world for every field the in-graph DR kernel writes, and for the eager
        # mass DR of Example 30.7 (body_mass/body_inertia + set_const, outside the graph):
        # batch the DERIVED fields that set_const writes too, or it writes one shared row (Ch 10)
        batch = ["actuator_gainprm", "dof_damping", "body_mass", "body_inertia",
                 *my_env.MASS_DERIVED]
        self.m = mjw.put_model(self.mjm, batch_sizes={k: nworld for k in batch})
        # no contacts (contype=conaffinity=0); the only constraint rows are the 2 slider limits
        self.d = mjw.make_data(self.mjm, nworld=nworld, nconmax=0, njmax=2)
        f32 = dict(dtype=float)
        self.action = wp.zeros((nworld, ACT_DIM), **f32)
        self.obs_wp = wp.zeros((nworld, OBS_DIM), **f32)
        self.final_obs_wp = wp.zeros((nworld, OBS_DIM), **f32)
        self.reward_wp = wp.zeros(nworld, **f32)
        self.terminated_wp = wp.zeros(nworld, dtype=bool)
        self.truncated_wp = wp.zeros(nworld, dtype=bool)
        self.done = wp.zeros(nworld, dtype=bool)
        self.t = wp.zeros(nworld, dtype=int)
        self.episode = wp.zeros(nworld, dtype=int)
        self.seed = wp.array([seed], dtype=int)      # device array: graphs read its CURRENT value
        self.ep_return = wp.zeros(nworld, **f32)
        self.last_return = wp.zeros(nworld, **f32)
        self.last_length = wp.zeros(nworld, dtype=int)
        self.n_finished = wp.zeros(nworld, dtype=int)
        self._all = wp.ones(nworld, dtype=bool)
        self.backend = backend
        self.graph = None
        self.use_graph = use_graph
        self._views()

    # ---- buffers seen by the learner -----------------------------------------------------------
    def _views(self):
        if self.backend == "torch":
            import torch                                   # only needed for this backend
            self._torch = torch
            self.obs = wp.to_torch(self.obs_wp)            # zero-copy views, made once
            self.final_obs = wp.to_torch(self.final_obs_wp)
            self.reward = wp.to_torch(self.reward_wp)
            self.terminated = wp.to_torch(self.terminated_wp)
            self.truncated = wp.to_torch(self.truncated_wp)
            self._action_t = wp.to_torch(self.action)
        elif self.backend == "warp":
            self.obs, self.final_obs, self.reward = self.obs_wp, self.final_obs_wp, self.reward_wp
            self.terminated, self.truncated = self.terminated_wp, self.truncated_wp
        else:
            raise ValueError(f"backend must be 'torch' or 'warp', got {self.backend!r}")

    # ---- the device-side pieces ----------------------------------------------------------------
    def _physics(self):
        wp.launch(_apply_action, dim=(self.nworld, ACT_DIM), inputs=[self.action],
                  outputs=[self.d.ctrl])
        for _ in range(self.n_substeps):
            mjw.step(self.m, self.d)
        wp.launch(_reward_done, dim=self.nworld,
                  inputs=[self.d.qpos, self.d.qvel, self.d.ctrl, self.swingup, self.max_steps,
                          1.7, float(np.cos(0.8)), self.t],
                  outputs=[self.reward_wp, self.terminated_wp, self.truncated_wp, self.done,
                           self.final_obs_wp])

    def _reset_where(self, mask):
        mjw.reset_data(self.m, self.d, reset=mask)       # qpos0, qvel 0, time 0, ... (masked)
        g, dmp = self.gain_range, self.damping_range
        wp.launch(_init_and_randomize, dim=self.nworld,
                  inputs=[self.seed, self.episode, mask, self.swingup, g[0], g[1], dmp[0], dmp[1]],
                  outputs=[self.d.qpos, self.d.qvel, self.m.actuator_gainprm,
                           self.m.dof_damping])

    def _observe(self, mask):
        wp.launch(_observe, dim=self.nworld,
                  inputs=[self.d.qpos, self.d.qvel, mask, self.reward_wp, self.t],
                  outputs=[self.obs_wp, self.ep_return, self.last_return, self.last_length,
                           self.n_finished])

    def _step_device(self):
        self._physics()
        self._reset_where(self.done)          # auto-reset only the finished worlds
        self._observe(self.done)

    # ---- public API ------------------------------------------------------------------------------
    def reset(self):
        """Reset every world (eager; call once, it also captures the graph). Returns obs."""
        if self.use_graph and self.graph is None:
            self._step_device()               # warm-up: compile every kernel before capturing
            if my_env.needs_conditional_fallback() and self.m.opt.graph_conditional:
                self.m.opt.graph_conditional = False      # CUDA < 12.4 (Ch 24)
            with wp.ScopedCapture() as cap:   # records the launches; runs nothing
                self._step_device()
            self.graph = cap.graph
        self._reset_where(self._all)
        self.reward_wp.zero_()
        self.done.zero_()
        self._observe(self.done)              # obs of the fresh state, no bookkeeping
        for a in (self.t, self.ep_return, self.last_return, self.last_length, self.n_finished):
            a.zero_()
        return self.obs

    def step(self, action):
        """Write the action (torch tensor or wp.array, shape (nworld, 1)), advance one period."""
        if self.backend == "torch":
            self._action_t.copy_(action.reshape(self.nworld, ACT_DIM))   # into the SAME buffer
        else:
            wp.copy(self.action, action)
        if self.graph is not None:
            wp.capture_launch(self.graph)
        else:
            self._step_device()
        return self.obs, self.reward, self.terminated, self.truncated

    def episode_stats(self):
        """Host summary (one sync): finished episodes, mean return/length of the latest ones."""
        n = self.n_finished.numpy()
        have = n > 0
        ret, length = self.last_return.numpy(), self.last_length.numpy()
        if not have.any():
            return {"episodes": 0, "mean_return": float("nan"), "mean_length": float("nan")}
        return {"episodes": int(n.sum()), "mean_return": float(ret[have].mean()),
                "mean_length": float(length[have].mean())}


if __name__ == "__main__":
    np.set_printoptions(precision=3, suppress=True)
    ON_GPU = wp.get_device().is_cuda
    NWORLD = 4096 if ON_GPU else 8
    try:
        import torch
        backend = "torch"
    except ImportError:
        backend = "warp"
        print("skipped: torch not installed, using backend='warp'")

    env = CartpoleEnv(NWORLD, backend=backend, seed=0)
    obs = env.reset()                       # warms up and captures the graph
    print("device:", wp.get_device(), "| backend:", backend, "| nworld:", NWORLD)
    print("obs:", type(obs).__module__.split(".")[0], tuple(obs.shape), "| graph captured:",
          env.graph is not None, f"| control period {env.dt:.2f} s")
    if backend == "torch":
        print("obs shares memory with the wp.array:", obs.data_ptr() == env.obs_wp.ptr)
        act = torch.zeros(NWORLD, 1)
    else:
        act = wp.zeros((NWORLD, 1), dtype=float)
    print("obs world 0:", np.asarray(obs[0].tolist() if backend == "torch" else obs.numpy()[0]))
    obs, rew, term, trunc = env.step(act)
    rew_np = rew.cpu().numpy() if backend == "torch" else rew.numpy()
    print("one step of rewards:", rew_np)
    print("motor gain per world:", env.m.actuator_gainprm.numpy()[:, 0, 0])
    print("slider damping per world:", env.m.dof_damping.numpy()[:, 0])
    print("terminated:", int(np.asarray(term.tolist()).sum()), "| truncated:",
          int(np.asarray(trunc.tolist()).sum()))
    print("overflow:", my_env.describe_overflow(env.d))

In [ ]:
%%writefile ppo_reference.py
"""ppo_reference.py - a small reference PPO for the on-device environments of Chapter 30.

Read it, run it, modify it; you do not need to write PPO from scratch. It works with any env that
has the interface of CartpoleEnv / HumanoidStandEnv with backend="torch":
  env.nworld, env.reset() -> obs, env.step(action) -> (obs, reward, terminated, truncated),
  env.final_obs (the observation a finished episode ended in), env.episode_stats().
The env tensors are zero-copy VIEWS of Warp buffers that the next env.step overwrites, so the
rollout clones what it keeps. Shapes: N = nworld, T = n_steps, D = obs dim, A = action dim.
"""
import time
from dataclasses import dataclass, field

import torch
import torch.nn as nn


@dataclass
class PPOConfig:
    n_steps: int = 32            # T: control steps per rollout (per world)
    gamma: float = 0.99          # discount
    lam: float = 0.95            # GAE lambda
    epochs: int = 8              # passes over each rollout
    minibatches: int = 4         # minibatches per pass
    clip: float = 0.2            # PPO ratio clip
    lr: float = 1e-3
    vf_coef: float = 0.5
    ent_coef: float = 0.0
    max_grad_norm: float = 1.0
    hidden: tuple = field(default=(64, 64))
    init_log_std: float = -1.0


class ActorCritic(nn.Module):
    """Gaussian policy and value function, each an MLP, with running observation normalization."""

    def __init__(self, obs_dim, act_dim, hidden=(64, 64), init_log_std=-1.0):
        super().__init__()

        def mlp(out):
            layers, d = [], obs_dim
            for h in hidden:
                layers += [nn.Linear(d, h), nn.Tanh()]
                d = h
            layers.append(nn.Linear(d, out))
            return nn.Sequential(*layers)

        self.pi, self.v = mlp(act_dim), mlp(1)
        nn.init.zeros_(self.pi[-1].bias)
        self.pi[-1].weight.data.mul_(0.01)            # start with near-zero actions
        self.log_std = nn.Parameter(torch.full((act_dim,), init_log_std))
        self.register_buffer("obs_mean", torch.zeros(obs_dim))
        self.register_buffer("obs_var", torch.ones(obs_dim))
        self.register_buffer("obs_count", torch.tensor(1e-4))

    @torch.no_grad()
    def update_norm(self, x):                          # parallel Welford update, x: (B, D)
        b_mean, b_var, b = x.mean(0), x.var(0, unbiased=False), x.shape[0]
        delta, tot = b_mean - self.obs_mean, self.obs_count + b
        self.obs_mean += delta * b / tot
        self.obs_var = (self.obs_var * self.obs_count + b_var * b
                        + delta ** 2 * self.obs_count * b / tot) / tot
        self.obs_count = tot

    def norm(self, obs):
        # the std floor (0.1) matters: a nearly constant input such as cos(theta) of a balanced
        # pole would otherwise be divided by ~0 and blow up exactly when the policy gets good
        return ((obs - self.obs_mean) / torch.sqrt(self.obs_var + 1e-2)).clamp(-10, 10)

    def dist(self, obs):
        mean = self.pi(self.norm(obs))
        return torch.distributions.Normal(mean, self.log_std.exp().expand_as(mean))

    def value(self, obs):
        return self.v(self.norm(obs)).squeeze(-1)

    @torch.no_grad()
    def act(self, obs, deterministic=True):
        d = self.dist(obs)
        return (d.mean if deterministic else d.sample()).clamp(-1.0, 1.0)


class PPO:
    def __init__(self, env, cfg=None, seed=0):
        self.env, self.cfg = env, cfg or PPOConfig()
        torch.manual_seed(seed)
        obs = env.reset()
        self.device = obs.device
        self.obs_dim, self.act_dim = obs.shape[1], env.action.shape[1]
        self.model = ActorCritic(self.obs_dim, self.act_dim, self.cfg.hidden,
                                 self.cfg.init_log_std).to(self.device)
        self.opt = torch.optim.Adam(self.model.parameters(), lr=self.cfg.lr)
        self._actor_params = list(self.model.pi.parameters()) + [self.model.log_std]
        self.obs = obs.clone()

    @torch.no_grad()
    def collect(self):
        """One rollout: T env steps in all N worlds at once. Returns (T, N, ...) tensors."""
        T, N, g = self.cfg.n_steps, self.env.nworld, self.cfg.gamma
        buf = {k: torch.zeros((T, N) + s, device=self.device) for k, s in
               [("obs", (self.obs_dim,)), ("act", (self.act_dim,)), ("logp", ()), ("val", ()),
                ("rew", ()), ("done", ())]}
        for t in range(T):
            dist = self.model.dist(self.obs)
            a = dist.sample()
            buf["obs"][t], buf["act"][t] = self.obs, a
            buf["logp"][t] = dist.log_prob(a).sum(-1)
            buf["val"][t] = self.model.value(self.obs)
            obs, rew, term, trunc = self.env.step(a.clamp(-1.0, 1.0))
            # time limit: the episode did NOT fail, so bootstrap its value. torch.where, not
            # `if trunc.any():` (a Python bool of a device tensor = a host sync every step), and
            # not `* trunc.float()` (NaN * 0 = NaN would leak from a diverged world's row).
            boot = torch.where(trunc, self.model.value(self.env.final_obs), torch.zeros_like(rew))
            buf["rew"][t] = rew + g * boot
            buf["done"][t] = (term | trunc).float()   # cut GAE at every episode boundary
            self.obs = obs.clone()                    # the view changes on the next step
        buf["last_val"] = self.model.value(self.obs)
        return buf

    @torch.no_grad()
    def gae(self, buf):
        """Generalized advantage estimation, backwards in time, all worlds in parallel."""
        T, g, lam = self.cfg.n_steps, self.cfg.gamma, self.cfg.lam
        adv = torch.zeros_like(buf["rew"])
        next_adv, next_val = torch.zeros_like(buf["last_val"]), buf["last_val"]
        for t in reversed(range(T)):
            keep = 1.0 - buf["done"][t]                     # 0 where an episode ended at t
            delta = buf["rew"][t] + g * next_val * keep - buf["val"][t]
            next_adv = delta + g * lam * keep * next_adv
            adv[t] = next_adv
            next_val = buf["val"][t]
        return adv, adv + buf["val"]                        # advantages, value targets

    def update(self, buf, adv, ret):
        """Clipped-surrogate PPO: `epochs` passes over shuffled minibatches of the T*N samples."""
        c = self.cfg
        flat = {k: buf[k].reshape((-1,) + buf[k].shape[2:]) for k in ("obs", "act", "logp")}
        adv, ret = adv.reshape(-1), ret.reshape(-1)
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)
        n = adv.shape[0]
        mb = max(1, n // c.minibatches)
        for _ in range(c.epochs):
            perm = torch.randperm(n, device=self.device)
            for i in range(0, n, mb):
                idx = perm[i:i + mb]
                dist = self.model.dist(flat["obs"][idx])
                logp = dist.log_prob(flat["act"][idx]).sum(-1)
                ratio = (logp - flat["logp"][idx]).exp()
                a = adv[idx]
                pg = -torch.min(ratio * a, ratio.clamp(1 - c.clip, 1 + c.clip) * a).mean()
                vf = 0.5 * (self.model.value(flat["obs"][idx]) - ret[idx]).pow(2).mean()
                ent = dist.entropy().sum(-1).mean()
                loss = pg + c.vf_coef * vf - c.ent_coef * ent
                self.opt.zero_grad()
                loss.backward()
                # clip actor and critic separately: the value loss (returns ~100) would
                # otherwise dominate one global norm and freeze the policy
                nn.utils.clip_grad_norm_(self._actor_params, c.max_grad_norm)
                nn.utils.clip_grad_norm_(self.model.v.parameters(), c.max_grad_norm)
                self.opt.step()
        return {"pg_loss": pg.item(), "v_loss": vf.item(), "entropy": ent.item()}

    def train(self, iters, log=print):
        history = []
        for it in range(iters):
            t0 = time.perf_counter()
            buf = self.collect()
            adv, ret = self.gae(buf)
            info = self.update(buf, adv, ret)
            self.model.update_norm(buf["obs"].reshape(-1, self.obs_dim))   # after the update,
            # so the log-probs stored in this rollout were computed with the same normalizer
            if self.device.type == "cuda":
                torch.cuda.synchronize()
            sec = time.perf_counter() - t0
            st = self.env.episode_stats()
            row = {"iter": it, "mean_return": st["mean_return"], "mean_length": st["mean_length"],
                   "episodes": st["episodes"],
                   "steps_per_s": self.cfg.n_steps * self.env.nworld / sec, **info}
            history.append(row)
            if log:
                log(f"iter {it:3d} | mean return {row['mean_return']:8.2f} | mean length "
                    f"{row['mean_length']:6.1f} | {row['steps_per_s']:9.0f} env-steps/s")
        return history

    def save(self, path):
        torch.save({"model": self.model.state_dict(), "obs_dim": self.obs_dim,
                    "act_dim": self.act_dim, "hidden": self.cfg.hidden}, path)


def load_policy(path, device="cpu"):
    """Load a checkpoint written by PPO.save; returns an ActorCritic in eval mode."""
    ck = torch.load(path, map_location=device)
    model = ActorCritic(ck["obs_dim"], ck["act_dim"], tuple(ck["hidden"])).to(device)
    model.load_state_dict(ck["model"])
    return model.eval()

In [ ]:
%%writefile ex10_humanoid_env.py
"""Example 30.10 - contact track, step 2: HumanoidStandEnv, the contact-rich environment.

Same architecture as CartpoleEnv (Example 30.1): one captured graph per env.step holding action
-> n_substeps x mjw.step -> reward/done kernel -> masked reset (mjw.reset_data + random joint
noise + per-episode floor-friction DR with my_env.randomize) -> observation kernel. New here:
  * buffer sizes measured by Example 30.9 (naconmax per world from the broadphase candidates,
    njmax from the worst row count, +25%) and the solver chosen there (Newton, 10 iterations);
  * overflow monitoring that survives auto-reset: reset_data CLEARS d.overflow of the worlds it
    resets, so the graph ORs d.overflow into env.overflow_seen before every reset, the device
    printf is silenced (m.opt.warn_overflow = 0) and env.overflow() decodes the sticky copy once
    per rollout with my_env.describe_overflow;
  * the contact-track dashboard in numbers: after every substep the graph keeps device-side
    running maxima of nacon (all worlds) and nefc (worst world) and sums solver_niter, and
    env.contact_stats() reads them once per rollout against naconmax / njmax;
  * observations from qpos/qvel, NOT from sensordata: sensors are evaluated inside mjw.step
    BEFORE the last integration and reset_data zeroes them (demonstrated below).
Other examples import this file: `from ex10_humanoid_env import HumanoidStandEnv`.
Run: python ex10_humanoid_env.py [--dump-json PATH]
  --dump-json PATH writes the torso height of every world per control step (page replay).
"""
import argparse
import json
import math
import pathlib
import sys
import types

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from etils import epath

HERE = pathlib.Path(__file__).parent if "__file__" in globals() else pathlib.Path.cwd()
sys.path.insert(0, str(HERE))
try:
    import my_env                           # notebook: written by the %%writefile cell
except ImportError:
    import my_env_step30 as my_env          # examples/30-capstone-rl-at-scale/

HUMANOID_PATH = epath.resource_path("mujoco_warp") / "test_data/humanoid/humanoid.xml"
NACON_PER_WORLD = 20     # Example 30.9 on CPU: 61 candidates / 4 worlds = 15.2, +25% -> 19.2
NJMAX = 43               # Example 30.9: worst 34 rows, +25%
HEIGHT_MIN = 0.8         # torso below this height = fallen (it starts at 1.28)


def humanoid_model():
    """The shipped humanoid (nq 28, nv 27, nu 21) plus three torso sensors added with MjSpec."""
    spec = mujoco.MjSpec.from_file(str(HUMANOID_PATH))
    xbody, body = mujoco.mjtObj.mjOBJ_XBODY, mujoco.mjtObj.mjOBJ_BODY   # XBODY = body frame;
    for name, kind, obj in [("torso_pos", mujoco.mjtSensor.mjSENS_FRAMEPOS, xbody),  # BODY would
                            ("torso_up", mujoco.mjtSensor.mjSENS_FRAMEZAXIS, xbody),  # be the COM
                            ("com_vel", mujoco.mjtSensor.mjSENS_SUBTREELINVEL, body)]:
        spec.add_sensor(name=name, type=kind, objtype=obj, objname="torso")
    return spec.compile()


@wp.kernel
def _apply_action(action: wp.array2d(dtype=float), ctrl: wp.array2d(dtype=float)):
    worldid, i = wp.tid()
    ctrl[worldid, i] = wp.clamp(action[worldid, i], -1.0, 1.0)


@wp.func
def _torso_up_z(qpos: wp.array2d(dtype=float), worldid: int) -> wp.vec3:
    # free joint: qpos[3:7] is the torso quaternion (w, x, y, z); return its body z axis
    w, x, y, z = qpos[worldid, 3], qpos[worldid, 4], qpos[worldid, 5], qpos[worldid, 6]
    return wp.vec3(2.0 * (x * z + w * y), 2.0 * (y * z - w * x), 1.0 - 2.0 * (x * x + y * y))


@wp.kernel
def _reward_done(
    qpos: wp.array2d(dtype=float), qvel: wp.array2d(dtype=float),
    ctrl: wp.array2d(dtype=float), max_steps: int, height_min: float,
    t: wp.array(dtype=int),
    reward: wp.array(dtype=float), terminated: wp.array(dtype=bool),
    truncated: wp.array(dtype=bool), done: wp.array(dtype=bool),
):
    worldid = wp.tid()
    steps = t[worldid] + 1
    t[worldid] = steps
    bad = bool(False)
    for i in range(qpos.shape[1]):
        if not wp.isfinite(qpos[worldid, i]):
            bad = True
    for i in range(qvel.shape[1]):
        if not wp.isfinite(qvel[worldid, i]):
            bad = True
    z = qpos[worldid, 2]
    up = _torso_up_z(qpos, worldid)
    ctrl_cost = float(0.0)
    for i in range(ctrl.shape[1]):
        ctrl_cost += ctrl[worldid, i] * ctrl[worldid, i]
    term = bad or z < height_min
    trunc = (not term) and steps >= max_steps
    r = float(0.0)
    if not bad:
        r = 2.0 * wp.min(z, 1.25) + up[2] - 0.01 * ctrl_cost     # height + upright - effort
    reward[worldid] = r
    terminated[worldid] = term
    truncated[worldid] = trunc
    done[worldid] = term or trunc


@wp.kernel
def _accumulate_overflow(overflow: wp.array(dtype=int), seen: wp.array(dtype=int)):
    worldid = wp.tid()
    seen[worldid] = seen[worldid] | overflow[worldid]       # before reset_data clears it


@wp.kernel
def _track_contacts(
    nacon: wp.array(dtype=int), nefc: wp.array(dtype=int), niter: wp.array(dtype=int),
    max_nacon: wp.array(dtype=int), max_nefc: wp.array(dtype=int),
    niter_sum: wp.array(dtype=int), nsub: wp.array(dtype=int),
):
    worldid = wp.tid()
    wp.atomic_max(max_nefc, 0, nefc[worldid])      # rows of the worst world
    wp.atomic_add(niter_sum, 0, niter[worldid])
    if worldid == 0:
        wp.atomic_max(max_nacon, 0, nacon[0])      # nacon counts contacts of ALL worlds
        nsub[0] = nsub[0] + 1


@wp.kernel
def _init_noise(
    seed: wp.array(dtype=int), episode: wp.array(dtype=int), reset: wp.array(dtype=bool),
    noise: float, qpos: wp.array2d(dtype=float), qvel: wp.array2d(dtype=float),
):
    worldid = wp.tid()
    if not reset[worldid]:
        return
    rng = wp.rand_init(seed[0] + 1, worldid * 7919 + episode[worldid] * 104729)
    for i in range(7, qpos.shape[1]):              # joints only; the free joint keeps qpos0
        qpos[worldid, i] = qpos[worldid, i] + wp.randf(rng, -noise, noise)
    for i in range(qvel.shape[1]):
        qvel[worldid, i] = wp.randf(rng, -noise, noise)


@wp.kernel
def _observe(
    qpos: wp.array2d(dtype=float), qvel: wp.array2d(dtype=float),
    done: wp.array(dtype=bool), reward: wp.array(dtype=float), t: wp.array(dtype=int),
    obs: wp.array2d(dtype=float), final_obs: wp.array2d(dtype=float), write_final: int,
    ep_return: wp.array(dtype=float), last_return: wp.array(dtype=float),
    last_length: wp.array(dtype=int), n_finished: wp.array(dtype=int),
):
    worldid = wp.tid()
    out = obs
    if write_final != 0:
        out = final_obs
    nq, nv = qpos.shape[1], qvel.shape[1]
    if write_final != 0:                           # a diverged world: zeros, never NaN,
        bad = bool(False)                          # so a NaN cannot reach the learner
        for i in range(nq):
            if not wp.isfinite(qpos[worldid, i]):
                bad = True
        for i in range(nv):
            if not wp.isfinite(qvel[worldid, i]):
                bad = True
        if bad:
            for i in range(final_obs.shape[1]):
                final_obs[worldid, i] = 0.0
            return
    up = _torso_up_z(qpos, worldid)
    out[worldid, 0] = qpos[worldid, 2]             # torso height
    out[worldid, 1] = up[0]
    out[worldid, 2] = up[1]
    out[worldid, 3] = up[2]
    for i in range(7, nq):                         # 21 joint angles
        out[worldid, i - 3] = qpos[worldid, i]
    for i in range(nv):                            # 27 velocities, scaled to O(1)
        out[worldid, nq - 3 + i] = 0.1 * qvel[worldid, i]
    if write_final != 0:
        return
    ret = ep_return[worldid] + reward[worldid]
    if done[worldid]:
        last_return[worldid] = ret
        last_length[worldid] = t[worldid]
        n_finished[worldid] = n_finished[worldid] + 1
        ret = 0.0
        t[worldid] = 0
    ep_return[worldid] = ret


class HumanoidStandEnv:
    """nworld humanoids that must keep standing; same interface as CartpoleEnv."""

    def __init__(self, nworld, backend="torch", n_substeps=4, max_steps=500, seed=0,
                 friction_range=(0.4, 1.0), noise=0.05, use_graph=True):
        self.nworld, self.n_substeps, self.max_steps = nworld, n_substeps, max_steps
        self.friction_range, self.noise = friction_range, noise
        self.mjm = humanoid_model()
        self.mjm.opt.solver = mujoco.mjtSolver.mjSOL_NEWTON          # Example 30.9 sweep
        self.mjm.opt.cone = mujoco.mjtCone.mjCONE_PYRAMIDAL
        self.mjm.opt.iterations = 10
        self.dt = n_substeps * float(self.mjm.opt.timestep)          # 0.02 s
        self.m = mjw.put_model(self.mjm, batch_sizes={"geom_friction": nworld})
        self.m.opt.warn_overflow = 0      # no device printf: the env decodes the bits itself
        self.d = mjw.make_data(self.mjm, nworld=nworld, njmax=NJMAX,
                               naconmax=math.ceil(NACON_PER_WORLD * nworld))
        self.obs_dim = 4 + (self.mjm.nq - 7) + self.mjm.nv            # 52
        f32 = dict(dtype=float)
        self.action = wp.zeros((nworld, self.mjm.nu), **f32)
        self.obs_wp = wp.zeros((nworld, self.obs_dim), **f32)
        self.final_obs_wp = wp.zeros((nworld, self.obs_dim), **f32)
        self.reward_wp = wp.zeros(nworld, **f32)
        self.terminated_wp = wp.zeros(nworld, dtype=bool)
        self.truncated_wp = wp.zeros(nworld, dtype=bool)
        self.done = wp.zeros(nworld, dtype=bool)
        self.t = wp.zeros(nworld, dtype=int)
        self.episode = wp.zeros(nworld, dtype=int)
        self.seed = wp.array([seed], dtype=int)
        self.ep_return = wp.zeros(nworld, **f32)
        self.last_return = wp.zeros(nworld, **f32)
        self.last_length = wp.zeros(nworld, dtype=int)
        self.n_finished = wp.zeros(nworld, dtype=int)
        self.overflow_seen = wp.zeros(nworld, dtype=int)
        self.stats = {k: wp.zeros(1, dtype=int) for k in ("max_nacon", "max_nefc", "niter_sum", "nsub")}
        self._all = wp.ones(nworld, dtype=bool)
        self.backend, self.graph, self.use_graph = backend, None, use_graph
        if backend == "torch":
            self.obs, self.final_obs = wp.to_torch(self.obs_wp), wp.to_torch(self.final_obs_wp)
            self.reward = wp.to_torch(self.reward_wp)
            self.terminated = wp.to_torch(self.terminated_wp)
            self.truncated = wp.to_torch(self.truncated_wp)
            self._action_t = wp.to_torch(self.action)
        else:
            self.obs, self.final_obs, self.reward = self.obs_wp, self.final_obs_wp, self.reward_wp
            self.terminated, self.truncated = self.terminated_wp, self.truncated_wp

    def _observe(self, mask, final=False):
        wp.launch(_observe, dim=self.nworld,
                  inputs=[self.d.qpos, self.d.qvel, mask, self.reward_wp, self.t],
                  outputs=[self.obs_wp, self.final_obs_wp, int(final), self.ep_return,
                           self.last_return, self.last_length, self.n_finished])

    def _reset_where(self, mask):
        mjw.reset_data(self.m, self.d, reset=mask)
        wp.launch(_init_noise, dim=self.nworld,
                  inputs=[self.seed, self.episode, mask, self.noise],
                  outputs=[self.d.qpos, self.d.qvel])
        lo, hi = self.friction_range                 # Ch 10 helper; also bumps episode[w]
        my_env.randomize(self.m, self.seed, self.episode, mask, lo, hi)

    def _step_device(self):
        wp.launch(_apply_action, dim=(self.nworld, self.mjm.nu), inputs=[self.action],
                  outputs=[self.d.ctrl])
        s = self.stats
        for _ in range(self.n_substeps):
            mjw.step(self.m, self.d)
            wp.launch(_track_contacts, dim=self.nworld,
                      inputs=[self.d.nacon, self.d.nefc, self.d.solver_niter],
                      outputs=[s["max_nacon"], s["max_nefc"], s["niter_sum"], s["nsub"]])
        wp.launch(_reward_done, dim=self.nworld,
                  inputs=[self.d.qpos, self.d.qvel, self.d.ctrl, self.max_steps, HEIGHT_MIN,
                          self.t],
                  outputs=[self.reward_wp, self.terminated_wp, self.truncated_wp, self.done])
        self._observe(self.done, final=True)        # the observation each episode ended in
        wp.launch(_accumulate_overflow, dim=self.nworld, inputs=[self.d.overflow],
                  outputs=[self.overflow_seen])
        self._reset_where(self.done)
        self._observe(self.done)

    def reset(self):
        if self.use_graph and self.graph is None:
            self._step_device()                      # warm-up compile
            if my_env.needs_conditional_fallback() and self.m.opt.graph_conditional:
                self.m.opt.graph_conditional = False
            with wp.ScopedCapture() as cap:
                self._step_device()
            self.graph = cap.graph
        self._reset_where(self._all)
        self.reward_wp.zero_()
        self.done.zero_()
        self._observe(self.done)
        for a in (self.t, self.ep_return, self.last_return, self.last_length, self.n_finished,
                  self.overflow_seen, *self.stats.values()):
            a.zero_()
        return self.obs

    def step(self, action):
        if self.backend == "torch":
            self._action_t.copy_(action.reshape(self.nworld, self.mjm.nu))
        else:
            wp.copy(self.action, action)
        if self.graph is not None:
            wp.capture_launch(self.graph)
        else:
            self._step_device()
        return self.obs, self.reward, self.terminated, self.truncated

    def episode_stats(self):
        n = self.n_finished.numpy()
        have = n > 0
        if not have.any():
            return {"episodes": 0, "mean_return": float("nan"), "mean_length": float("nan")}
        return {"episodes": int(n.sum()),
                "mean_return": float(self.last_return.numpy()[have].mean()),
                "mean_length": float(self.last_length.numpy()[have].mean())}

    def overflow(self, clear=True):
        """Decode every overflow bit any world raised since the last call (one host sync)."""
        seen = wp.clone(self.overflow_seen)
        wp.launch(_accumulate_overflow, dim=self.nworld, inputs=[self.d.overflow], outputs=[seen])
        text = my_env.describe_overflow(types.SimpleNamespace(overflow=seen))
        if clear:
            self.overflow_seen.zero_()
        return text

    def contact_stats(self, clear=True):
        """Worst nacon and nefc and the mean solver_niter over all substeps since the last call."""
        v = {k: int(a.numpy()[0]) for k, a in self.stats.items()}
        if clear:
            for a in self.stats.values():
                a.zero_()                           # contents only: the graph keeps its arrays
        mean_niter = v["niter_sum"] / max(1, v["nsub"] * self.nworld)
        return {"max_nacon": v["max_nacon"], "naconmax": self.d.naconmax,
                "max_nefc": v["max_nefc"], "njmax": self.d.njmax, "mean_niter": mean_niter,
                "text": f"max nacon {v['max_nacon']}/{self.d.naconmax}, max nefc "
                        f"{v['max_nefc']}/{self.d.njmax}, mean niter {mean_niter:.2f}"}


if __name__ == "__main__":
    np.set_printoptions(precision=3, suppress=True)
    parser = argparse.ArgumentParser()
    parser.add_argument("--dump-json", default=None)
    args, _ = parser.parse_known_args()
    ON_GPU = wp.get_device().is_cuda
    NWORLD = 1024 if ON_GPU else 4
    env = HumanoidStandEnv(NWORLD, backend="warp", seed=0)
    obs = env.reset()
    print("device:", wp.get_device(), "| nworld:", NWORLD, f"| naconmax {env.d.naconmax}, "
          f"njmax {env.d.njmax} | solver Newton, pyramidal, 10 iterations")
    print("obs shape:", obs.shape, "| action shape:", env.action.shape,
          f"| control period {env.dt:.2f} s")
    print("floor friction per world:", env.m.geom_friction.numpy()[:, 0, 0].round(4))
    act = wp.zeros((NWORLD, env.mjm.nu), dtype=float)       # zero torque: it will slump
    obs, rew, term, trunc = env.step(act)
    print("one step of rewards:", rew.numpy())
    heights = [env.d.qpos.numpy()[:, 2].round(3).tolist()]
    for _ in range(149):                                    # 3 s with zero actions
        env.step(act)
        heights.append(env.d.qpos.numpy()[:, 2].round(3).tolist())   # demo only: host read
    st = env.episode_stats()
    print(f"after 3 s of zero torque: {st['episodes']} falls, mean episode length "
          f"{st['mean_length']:.1f} steps, episodes per world {env.n_finished.numpy()}")
    print("overflow seen during the episodes:", env.overflow().replace("\n", "; "))
    print("contact budget during the episodes:", env.contact_stats()["text"])

    # why the observations come from qpos and not from sensordata
    sl = my_env.sensor_slices(env.mjm)
    print("sensor slices:", sl)
    adr = sl["torso_pos"][0] + 2                            # z of the torso_pos sensor
    m2, d2 = env.m, mjw.make_data(env.mjm, nworld=NWORLD, njmax=NJMAX,
                                  naconmax=math.ceil(NACON_PER_WORLD * NWORLD))
    d2.qvel.assign(np.tile(np.r_[0.0, 0.0, -1.0, np.zeros(env.mjm.nv - 3)], (NWORLD, 1)))
    mjw.step(m2, d2)                                        # torso moving down at 1 m/s
    print(f"after one mjw.step: torso z from qpos {d2.qpos.numpy()[0, 2]:.4f}, from sensordata "
          f"{d2.sensordata.numpy()[0, adr]:.4f} (= z before this step's integration)")
    mjw.reset_data(m2, d2)
    print("after reset_data: sensordata torso_pos =", d2.sensordata.numpy()[0, :3],
          "(zero until the next forward/step)")
    if args.dump_json:
        with open(args.dump_json, "w") as f:
            json.dump({"nworld": NWORLD, "dt": 2 * env.dt, "kind": "height", "zmin": 0.0,
                       "zmax": 1.4, "frames": heights[::2]},   # dt: one frame per 2 steps
                      f, separators=(",", ":"))
        print("wrote", args.dump_json, len(heights[::2]), "frames")

## Warm-up track
### 4. Implementing CartpoleEnv
One `env.step(action)` = copy the action in, then replay ONE captured graph: action -> substeps -> reward/done -> masked reset + in-graph DR -> observation. The cell below runs the same file as the `%%writefile ex1_env.py` cell, as a script.

In [ ]:
# Example 30.1 - ex1_env.py
"""Example 30.1 - CartpoleEnv: a fully on-device RL environment for thousands of worlds.

One call of env.step(action) replays ONE captured graph that does everything on the device:
  1. copy the policy's action into d.ctrl (clamped),
  2. n_substeps physics steps (mjw.step),
  3. reward / termination / truncation kernel (with a non-finite guard),
  4. masked auto-reset of the finished worlds: mjw.reset_data, a random initial state, and an
     in-graph domain-randomization kernel (motor gain, slider damping) for the NEW episode,
  5. the observation kernel and the episode bookkeeping (returns and lengths).
The host never reads a value inside the loop. backend="torch" hands out zero-copy torch tensors
(Ch 28); backend="warp" hands out the wp.arrays themselves, so the env also works without torch.
Other examples import this file: `from ex1_env import CartpoleEnv`.
Run: python ex1_env.py
"""
import pathlib
import sys

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

HERE = pathlib.Path(__file__).parent if "__file__" in globals() else pathlib.Path.cwd()
sys.path.insert(0, str(HERE))
try:
    import my_env                           # notebook: written by the %%writefile cell
except ImportError:
    import my_env_step30 as my_env          # examples/30-capstone-rl-at-scale/

CARTPOLE = """
<mujoco model="cartpole">
  <option timestep="0.01" integrator="implicitfast"/>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="front" pos="0 -2 2.1" xyaxes="1 0 0 0 1 2"/>
    <geom name="floor" type="plane" size="20 20 .1" rgba=".75 .8 .85 1" contype="0" conaffinity="0"/>
    <geom name="rail" type="capsule" fromto="-2 0 1 2 0 1" size=".02" contype="0" conaffinity="0"/>
    <body name="cart" pos="0 0 1">
      <joint name="slider" type="slide" axis="1 0 0" range="-1.8 1.8" limited="true" damping="0.1"/>
      <geom name="cart" type="box" size=".2 .1 .05" mass="1" rgba=".15 .3 .7 1" contype="0" conaffinity="0"/>
      <body name="pole">
        <joint name="hinge" type="hinge" axis="0 1 0" damping="0.01"/>
        <geom name="pole" type="capsule" fromto="0 0 0 0 0 .6" size=".03" mass="0.1" rgba=".9 .45 .1 1"
              contype="0" conaffinity="0"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="push" joint="slider" gear="10" ctrlrange="-1 1" ctrllimited="true"/></actuator>
  <sensor>
    <jointpos joint="slider"/><jointpos joint="hinge"/>
    <jointvel joint="slider"/><jointvel joint="hinge"/>
  </sensor>
</mujoco>
"""

vec10 = wp.types.vector(length=10, dtype=float)    # the dtype of m.actuator_gainprm
OBS_DIM = 5          # x, cos(theta), sin(theta), x_dot, theta_dot
ACT_DIM = 1


@wp.kernel
def _apply_action(action: wp.array2d(dtype=float), ctrl: wp.array2d(dtype=float)):
    worldid, i = wp.tid()
    ctrl[worldid, i] = wp.clamp(action[worldid, i], -1.0, 1.0)


@wp.kernel
def _reward_done(
    qpos: wp.array2d(dtype=float), qvel: wp.array2d(dtype=float),
    ctrl: wp.array2d(dtype=float),
    swingup: int, max_steps: int, x_limit: float, cos_limit: float,
    t: wp.array(dtype=int),
    reward: wp.array(dtype=float), terminated: wp.array(dtype=bool),
    truncated: wp.array(dtype=bool), done: wp.array(dtype=bool),
    final_obs: wp.array2d(dtype=float),
):
    worldid = wp.tid()
    x = qpos[worldid, 0]
    th = qpos[worldid, 1]
    xd = qvel[worldid, 0]
    thd = qvel[worldid, 1]
    a = ctrl[worldid, 0]
    steps = t[worldid] + 1
    t[worldid] = steps
    # non-finite guard: MJWarp has no mj_checkPos/mj_checkVel, so the env checks itself (Ch 8)
    bad = not (wp.isfinite(x) and wp.isfinite(th) and wp.isfinite(xd) and wp.isfinite(thd))
    c = wp.cos(th)
    term = bad or wp.abs(x) > x_limit
    if swingup == 0:
        term = term or c < cos_limit              # balance task: the pole fell
    trunc = (not term) and steps >= max_steps     # time limit: NOT a failure
    r = float(0.0)
    if not bad:
        if swingup == 0:
            r = c - 0.25 * x * x - 0.01 * a * a     # upright, centred, gentle
        else:
            # energy shaping: pole energy (normalized so upright at rest = 1) should reach 1.
            # Kept >= about -0.15 so that crashing into the rail end never pays off.
            e = (0.5 * 0.012 * thd * thd + 0.2943 * c) / 0.2943
            r = 0.5 * (1.0 + c) + 0.5 * wp.exp(-2.0 * wp.abs(e - 1.0))
            r = r - 0.05 * x * x - 0.01 * a * a
    reward[worldid] = r
    terminated[worldid] = term
    truncated[worldid] = trunc
    done[worldid] = term or trunc
    # the observation of the state the episode ENDED in (needed to bootstrap truncations)
    final_obs[worldid, 0] = x
    final_obs[worldid, 1] = c
    final_obs[worldid, 2] = wp.sin(th)
    final_obs[worldid, 3] = xd
    final_obs[worldid, 4] = thd
    if bad:
        for i in range(5):
            final_obs[worldid, i] = 0.0


@wp.kernel
def _init_and_randomize(
    seed: wp.array(dtype=int), episode: wp.array(dtype=int), reset: wp.array(dtype=bool),
    swingup: int, gain_lo: float, gain_hi: float, damp_lo: float, damp_hi: float,
    qpos: wp.array2d(dtype=float), qvel: wp.array2d(dtype=float),
    actuator_gainprm: wp.array2d(dtype=vec10),
    dof_damping: wp.array2d(dtype=float),
):
    worldid = wp.tid()
    if not reset[worldid]:
        return
    # one random stream per (seed, world, episode): reproducible, new for every episode
    rng = wp.rand_init(seed[0], worldid * 7919 + episode[worldid] * 104729)
    # random initial state (reset_data put qpos0 = upright, qvel = 0)
    qpos[worldid, 0] = wp.randf(rng, -0.2, 0.2)
    th0 = wp.randf(rng, -0.1, 0.1)
    if swingup != 0:
        th0 = th0 + wp.pi
    qpos[worldid, 1] = th0
    qvel[worldid, 0] = wp.randf(rng, -0.1, 0.1)
    qvel[worldid, 1] = wp.randf(rng, -0.1, 0.1)
    # in-graph domain randomization: fields with NO derived quantities (no set_const needed)
    g = actuator_gainprm[worldid, 0]
    g[0] = wp.randf(rng, gain_lo, gain_hi)        # motor gain (force = gain * gear * ctrl)
    actuator_gainprm[worldid, 0] = g
    dof_damping[worldid, 0] = wp.randf(rng, damp_lo, damp_hi)     # slider damping
    episode[worldid] = episode[worldid] + 1


@wp.kernel
def _observe(
    qpos: wp.array2d(dtype=float), qvel: wp.array2d(dtype=float),
    done: wp.array(dtype=bool), reward: wp.array(dtype=float), t: wp.array(dtype=int),
    obs: wp.array2d(dtype=float),
    ep_return: wp.array(dtype=float), last_return: wp.array(dtype=float),
    last_length: wp.array(dtype=int), n_finished: wp.array(dtype=int),
):
    worldid = wp.tid()
    th = qpos[worldid, 1]
    obs[worldid, 0] = qpos[worldid, 0]
    obs[worldid, 1] = wp.cos(th)
    obs[worldid, 2] = wp.sin(th)
    obs[worldid, 3] = qvel[worldid, 0]
    obs[worldid, 4] = qvel[worldid, 1]
    ret = ep_return[worldid] + reward[worldid]
    if done[worldid]:                    # bookkeeping on the device: no host sync per step
        last_return[worldid] = ret
        last_length[worldid] = t[worldid]
        n_finished[worldid] = n_finished[worldid] + 1
        ret = 0.0
        t[worldid] = 0
    ep_return[worldid] = ret


class CartpoleEnv:
    """nworld cartpoles stepped as one batch. step() is one graph replay (or eager calls).

    obs (nworld, 5), reward (nworld,), terminated / truncated (nworld,) bool, final_obs (nworld, 5).
    Finished worlds are reset INSIDE step(): the returned obs of a finished world is already the
    first observation of its next episode, and final_obs holds the observation it ended in.
    """

    def __init__(self, nworld, backend="torch", n_substeps=2, max_steps=200, seed=0,
                 swingup=False, gain_range=(0.8, 1.2), damping_range=(0.05, 0.2),
                 use_graph=True):
        self.nworld, self.n_substeps, self.max_steps = nworld, n_substeps, max_steps
        self.swingup = int(swingup)
        self.gain_range, self.damping_range = gain_range, damping_range
        self.mjm = mujoco.MjModel.from_xml_string(CARTPOLE)
        self.dt = n_substeps * float(self.mjm.opt.timestep)          # control period [s]
        # one row per world for every field the in-graph DR kernel writes, and for the eager
        # mass DR of Example 30.7 (body_mass/body_inertia + set_const, outside the graph):
        # batch the DERIVED fields that set_const writes too, or it writes one shared row (Ch 10)
        batch = ["actuator_gainprm", "dof_damping", "body_mass", "body_inertia",
                 *my_env.MASS_DERIVED]
        self.m = mjw.put_model(self.mjm, batch_sizes={k: nworld for k in batch})
        # no contacts (contype=conaffinity=0); the only constraint rows are the 2 slider limits
        self.d = mjw.make_data(self.mjm, nworld=nworld, nconmax=0, njmax=2)
        f32 = dict(dtype=float)
        self.action = wp.zeros((nworld, ACT_DIM), **f32)
        self.obs_wp = wp.zeros((nworld, OBS_DIM), **f32)
        self.final_obs_wp = wp.zeros((nworld, OBS_DIM), **f32)
        self.reward_wp = wp.zeros(nworld, **f32)
        self.terminated_wp = wp.zeros(nworld, dtype=bool)
        self.truncated_wp = wp.zeros(nworld, dtype=bool)
        self.done = wp.zeros(nworld, dtype=bool)
        self.t = wp.zeros(nworld, dtype=int)
        self.episode = wp.zeros(nworld, dtype=int)
        self.seed = wp.array([seed], dtype=int)      # device array: graphs read its CURRENT value
        self.ep_return = wp.zeros(nworld, **f32)
        self.last_return = wp.zeros(nworld, **f32)
        self.last_length = wp.zeros(nworld, dtype=int)
        self.n_finished = wp.zeros(nworld, dtype=int)
        self._all = wp.ones(nworld, dtype=bool)
        self.backend = backend
        self.graph = None
        self.use_graph = use_graph
        self._views()

    # ---- buffers seen by the learner -----------------------------------------------------------
    def _views(self):
        if self.backend == "torch":
            import torch                                   # only needed for this backend
            self._torch = torch
            self.obs = wp.to_torch(self.obs_wp)            # zero-copy views, made once
            self.final_obs = wp.to_torch(self.final_obs_wp)
            self.reward = wp.to_torch(self.reward_wp)
            self.terminated = wp.to_torch(self.terminated_wp)
            self.truncated = wp.to_torch(self.truncated_wp)
            self._action_t = wp.to_torch(self.action)
        elif self.backend == "warp":
            self.obs, self.final_obs, self.reward = self.obs_wp, self.final_obs_wp, self.reward_wp
            self.terminated, self.truncated = self.terminated_wp, self.truncated_wp
        else:
            raise ValueError(f"backend must be 'torch' or 'warp', got {self.backend!r}")

    # ---- the device-side pieces ----------------------------------------------------------------
    def _physics(self):
        wp.launch(_apply_action, dim=(self.nworld, ACT_DIM), inputs=[self.action],
                  outputs=[self.d.ctrl])
        for _ in range(self.n_substeps):
            mjw.step(self.m, self.d)
        wp.launch(_reward_done, dim=self.nworld,
                  inputs=[self.d.qpos, self.d.qvel, self.d.ctrl, self.swingup, self.max_steps,
                          1.7, float(np.cos(0.8)), self.t],
                  outputs=[self.reward_wp, self.terminated_wp, self.truncated_wp, self.done,
                           self.final_obs_wp])

    def _reset_where(self, mask):
        mjw.reset_data(self.m, self.d, reset=mask)       # qpos0, qvel 0, time 0, ... (masked)
        g, dmp = self.gain_range, self.damping_range
        wp.launch(_init_and_randomize, dim=self.nworld,
                  inputs=[self.seed, self.episode, mask, self.swingup, g[0], g[1], dmp[0], dmp[1]],
                  outputs=[self.d.qpos, self.d.qvel, self.m.actuator_gainprm,
                           self.m.dof_damping])

    def _observe(self, mask):
        wp.launch(_observe, dim=self.nworld,
                  inputs=[self.d.qpos, self.d.qvel, mask, self.reward_wp, self.t],
                  outputs=[self.obs_wp, self.ep_return, self.last_return, self.last_length,
                           self.n_finished])

    def _step_device(self):
        self._physics()
        self._reset_where(self.done)          # auto-reset only the finished worlds
        self._observe(self.done)

    # ---- public API ------------------------------------------------------------------------------
    def reset(self):
        """Reset every world (eager; call once, it also captures the graph). Returns obs."""
        if self.use_graph and self.graph is None:
            self._step_device()               # warm-up: compile every kernel before capturing
            if my_env.needs_conditional_fallback() and self.m.opt.graph_conditional:
                self.m.opt.graph_conditional = False      # CUDA < 12.4 (Ch 24)
            with wp.ScopedCapture() as cap:   # records the launches; runs nothing
                self._step_device()
            self.graph = cap.graph
        self._reset_where(self._all)
        self.reward_wp.zero_()
        self.done.zero_()
        self._observe(self.done)              # obs of the fresh state, no bookkeeping
        for a in (self.t, self.ep_return, self.last_return, self.last_length, self.n_finished):
            a.zero_()
        return self.obs

    def step(self, action):
        """Write the action (torch tensor or wp.array, shape (nworld, 1)), advance one period."""
        if self.backend == "torch":
            self._action_t.copy_(action.reshape(self.nworld, ACT_DIM))   # into the SAME buffer
        else:
            wp.copy(self.action, action)
        if self.graph is not None:
            wp.capture_launch(self.graph)
        else:
            self._step_device()
        return self.obs, self.reward, self.terminated, self.truncated

    def episode_stats(self):
        """Host summary (one sync): finished episodes, mean return/length of the latest ones."""
        n = self.n_finished.numpy()
        have = n > 0
        ret, length = self.last_return.numpy(), self.last_length.numpy()
        if not have.any():
            return {"episodes": 0, "mean_return": float("nan"), "mean_length": float("nan")}
        return {"episodes": int(n.sum()), "mean_return": float(ret[have].mean()),
                "mean_length": float(length[have].mean())}


if __name__ == "__main__":
    np.set_printoptions(precision=3, suppress=True)
    ON_GPU = wp.get_device().is_cuda
    NWORLD = 4096 if ON_GPU else 8
    try:
        import torch
        backend = "torch"
    except ImportError:
        backend = "warp"
        print("skipped: torch not installed, using backend='warp'")

    env = CartpoleEnv(NWORLD, backend=backend, seed=0)
    obs = env.reset()                       # warms up and captures the graph
    print("device:", wp.get_device(), "| backend:", backend, "| nworld:", NWORLD)
    print("obs:", type(obs).__module__.split(".")[0], tuple(obs.shape), "| graph captured:",
          env.graph is not None, f"| control period {env.dt:.2f} s")
    if backend == "torch":
        print("obs shares memory with the wp.array:", obs.data_ptr() == env.obs_wp.ptr)
        act = torch.zeros(NWORLD, 1)
    else:
        act = wp.zeros((NWORLD, 1), dtype=float)
    print("obs world 0:", np.asarray(obs[0].tolist() if backend == "torch" else obs.numpy()[0]))
    obs, rew, term, trunc = env.step(act)
    rew_np = rew.cpu().numpy() if backend == "torch" else rew.numpy()
    print("one step of rewards:", rew_np)
    print("motor gain per world:", env.m.actuator_gainprm.numpy()[:, 0, 0])
    print("slider damping per world:", env.m.dof_damping.numpy()[:, 0])
    print("terminated:", int(np.asarray(term.tolist()).sum()), "| truncated:",
          int(np.asarray(trunc.tolist()).sum()))
    print("overflow:", my_env.describe_overflow(env.d))

Eager launches versus one graph replay per control step (your own numbers):

In [ ]:
# Example 30.2 - ex2_random_policy.py
"""Example 30.2 - random actions; env steps/s with eager launches vs one graph replay per step.

Both envs run the same device code (Example 30.1); only the dispatch differs: eager issues every
kernel launch from Python on every step, the graph replays the recorded launches with one call.
The random actions are drawn on the device (torch) and written into the env's action buffer.
Timing: warm up, then wp.synchronize() before reading the clock. On CPU these are numbers from
Warp's CPU device ("on a CPU dev box"), NOT GPU performance; on a GPU print your own numbers.
Run: python ex2_random_policy.py
"""
import pathlib
import sys
import time

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

HERE = pathlib.Path(__file__).parent if "__file__" in globals() else pathlib.Path.cwd()
sys.path.insert(0, str(HERE))
from ex1_env import CartpoleEnv

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 1000 if ON_GPU else 50

try:
    import torch
    BACKEND = "torch"
except ImportError:
    torch = None
    BACKEND = "warp"
    print("skipped: torch not installed, using backend='warp' with NumPy actions")

rng = np.random.default_rng(0)
print("device:", wp.get_device(), "| nworld:", NWORLD, "| env steps:", NSTEP, "| backend:", BACKEND)
for use_graph in (False, True):
    env = CartpoleEnv(NWORLD, backend=BACKEND, seed=1, use_graph=use_graph)
    env.reset()
    if torch is not None:
        gen = torch.Generator(device=env.obs.device).manual_seed(0)
        actions = [torch.rand(NWORLD, 1, generator=gen, device=env.obs.device) * 2 - 1
                   for _ in range(NSTEP)]
    else:
        actions = [wp.array(rng.uniform(-1, 1, (NWORLD, 1)).astype(np.float32))
                   for _ in range(NSTEP)]
    env.step(actions[0])                     # warm-up (eager path compiles here)
    wp.synchronize()
    t0 = time.perf_counter()
    for a in actions:
        env.step(a)
    wp.synchronize()
    sec = time.perf_counter() - t0
    st = env.episode_stats()
    mode = "graph" if use_graph else "eager"
    print(f"{mode:5s}: {NSTEP * NWORLD / sec:10.0f} env-steps/s "
          f"({NSTEP * NWORLD * env.n_substeps / sec:10.0f} physics world-steps/s) | "
          f"episodes finished {st['episodes']:3d}, mean length {st['mean_length']:.1f}")
print(f"a random policy keeps the pole up for only {st['mean_length'] * env.dt:.2f} s on average "
      f"(the time limit is {env.max_steps * env.dt:.0f} s)")

Test the reward, termination and observation kernels against NumPy before training anything:

In [ ]:
# Example 30.3 - ex3_reward_check.py
"""Example 30.3 - test the reward / termination / observation kernels against NumPy.

Writes random states (some far outside the safe set, one NaN) into d.qpos / d.qvel / d.ctrl,
launches the env's _reward_done and _observe kernels directly, and compares every output with a
NumPy reference written from the task definition. Do this for every kernel you write: a sign
error in a reward trains a policy that is perfect at the wrong task.
Run: python ex3_reward_check.py
"""
import pathlib
import sys

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

HERE = pathlib.Path(__file__).parent if "__file__" in globals() else pathlib.Path.cwd()
sys.path.insert(0, str(HERE))
import ex1_env
from ex1_env import CartpoleEnv

NWORLD = 64
X_LIMIT, TH_LIMIT, MAX_STEPS = 1.7, 0.8, 200


def reference(qpos, qvel, ctrl, t, swingup):
    """The task, written once more in NumPy (float64)."""
    x, th, xd, thd, a = qpos[:, 0], qpos[:, 1], qvel[:, 0], qvel[:, 1], ctrl[:, 0]
    bad = ~(np.isfinite(qpos).all(1) & np.isfinite(qvel).all(1))
    c = np.cos(th)
    term = bad | (np.abs(x) > X_LIMIT)
    if not swingup:
        term |= c < np.cos(TH_LIMIT)
        r = c - 0.25 * x**2 - 0.01 * a**2
    else:
        e = (0.5 * 0.012 * thd**2 + 0.2943 * c) / 0.2943
        r = 0.5 * (1 + c) + 0.5 * np.exp(-2 * np.abs(e - 1)) - 0.05 * x**2 - 0.01 * a**2
    r = np.where(bad, 0.0, r)
    trunc = ~term & (t + 1 >= MAX_STEPS)
    obs = np.stack([x, c, np.sin(th), xd, thd], 1)
    return r, term, trunc, obs


rng = np.random.default_rng(0)
print("device:", wp.get_device(), "| random states:", NWORLD)
for swingup in (False, True):
    env = CartpoleEnv(NWORLD, backend="warp", swingup=swingup, use_graph=False)
    qpos = np.stack([rng.uniform(-2.0, 2.0, NWORLD), rng.uniform(-np.pi, np.pi, NWORLD)], 1)
    qvel = rng.normal(0.0, 2.0, (NWORLD, 2))
    ctrl = rng.uniform(-1, 1, (NWORLD, 1))
    qpos[3, 1] = np.nan                                  # one broken world
    t = rng.integers(0, MAX_STEPS, NWORLD)
    t[:4] = MAX_STEPS - 1                                # a few worlds hit the time limit
    env.d.qpos.assign(qpos.astype(np.float32))
    env.d.qvel.assign(qvel.astype(np.float32))
    env.d.ctrl.assign(ctrl.astype(np.float32))
    env.t.assign(t.astype(np.int32))
    wp.launch(ex1_env._reward_done, dim=NWORLD,
              inputs=[env.d.qpos, env.d.qvel, env.d.ctrl, int(swingup), MAX_STEPS, X_LIMIT,
                      float(np.cos(TH_LIMIT)), env.t],
              outputs=[env.reward_wp, env.terminated_wp, env.truncated_wp, env.done,
                       env.final_obs_wp])
    env._observe(env.done)
    # float32 inputs, as the kernels see them
    r, term, trunc, obs = reference(qpos.astype(np.float32).astype(np.float64),
                                    qvel.astype(np.float32).astype(np.float64),
                                    ctrl.astype(np.float32).astype(np.float64), t, swingup)
    ok = np.isfinite(obs).all(1)
    name = "swing-up" if swingup else "balance "
    print(f"{name}: max |reward diff| = {np.abs(env.reward_wp.numpy() - r).max():.2e} | "
          f"max |obs diff| = {np.abs(env.obs_wp.numpy()[ok] - obs[ok]).max():.2e}")
    print(f"          terminated match: {bool((env.terminated_wp.numpy() == term).all())} "
          f"({int(term.sum())} worlds) | truncated match: "
          f"{bool((env.truncated_wp.numpy() == trunc).all())} ({int(trunc.sum())} worlds)")
    print(f"          NaN world 3: terminated={bool(env.terminated_wp.numpy()[3])}, "
          f"reward={env.reward_wp.numpy()[3]:.1f}, final_obs zeroed="
          f"{bool((env.final_obs_wp.numpy()[3] == 0).all())}")

### 5. Asynchronous episodes and auto-reset
Termination (failure) and truncation (time limit) are reported separately; every world restarts on its own schedule.

In [ ]:
# Example 30.4 - ex4_episodes.py
"""Example 30.4 - episodes end and restart per world; truncation is flagged apart from termination.

A noisy linear controller (gains from an LQR design, scaled differently per world) runs 16
worlds for 300 control steps with a 100-step time limit. Strong controllers survive to the time
limit (TRUNCATED: the episode was cut, the pole did not fall); weak ones let the pole fall
(TERMINATED). Every world restarts on its own schedule (asynchronous episodes).
Run: python ex4_episodes.py [--dump-json PATH]
  --dump-json PATH writes the per-world episode segments for the page's episode timeline.
"""
import argparse
import json
import pathlib
import sys

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

HERE = pathlib.Path(__file__).parent if "__file__" in globals() else pathlib.Path.cwd()
sys.path.insert(0, str(HERE))
from ex1_env import CartpoleEnv

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

NWORLD, NSTEP, MAX_STEPS = 16, 300, 100
K = np.array([0.9, 6.8, 1.1, 1.2])          # LQR-like gains on (x, theta, x_dot, theta_dot)
scale = np.linspace(0.0, 1.0, NWORLD)       # world k uses scale[k] * K (world 0: no control)

env = CartpoleEnv(NWORLD, backend="warp", max_steps=MAX_STEPS, seed=3)
env.reset()
rng = np.random.default_rng(0)
act = wp.zeros((NWORLD, 1), dtype=float)
events = []                                 # (step, world, "T" | "R")
for i in range(NSTEP):
    # this demo reads obs on the host every step to log events; a training loop never does
    o = env.obs_wp.numpy()
    th = np.arctan2(o[:, 2], o[:, 1])
    a = scale * (np.stack([o[:, 0], th, o[:, 3], o[:, 4]], 1) @ K) + rng.normal(0, 0.3, NWORLD)
    act.assign(a.reshape(NWORLD, 1).astype(np.float32))
    env.step(act)
    term, trunc = env.terminated_wp.numpy(), env.truncated_wp.numpy()
    assert not (term & trunc).any(), "a step is either terminated or truncated, never both"
    for w in np.nonzero(term)[0]:
        events.append((i + 1, int(w), "T"))
    for w in np.nonzero(trunc)[0]:
        events.append((i + 1, int(w), "R"))

lengths = {"T": [], "R": []}
segments = [[] for _ in range(NWORLD)]
last = np.zeros(NWORLD, dtype=int)
for step, w, kind in events:
    lengths[kind].append(step - last[w])
    segments[w].append([int(last[w]), int(step), kind])
    last[w] = step
for w in range(NWORLD):
    if last[w] < NSTEP:
        segments[w].append([int(last[w]), NSTEP, "open"])
print("device:", wp.get_device(), "| nworld:", NWORLD, "| control steps:", NSTEP,
      "| time limit:", MAX_STEPS)
print(f"terminated episodes: {len(lengths['T'])} | truncated episodes: {len(lengths['R'])}")
print("episodes per world:", [len([s for s in seg if s[2] != 'open']) for seg in segments])
bins = [0, 20, 40, 60, 80, 100, 101]                 # last bin = exactly 100 steps
cT, edges = np.histogram(lengths["T"], bins=bins)   # separate counts: a TERMINATED episode
cR, _ = np.histogram(lengths["R"], bins=bins)       # can also last exactly 100 steps
print("  episode lengths (#: terminated, =: truncated)")
for lo, hi, t, r in zip(edges[:-1], edges[1:], cT, cR):
    label = f"{lo:3d}-{hi - 1:3d}" if hi - lo > 1 else f"{lo:3d}    "
    print(f"  length {label}: {'#' * t}{'=' * r} {t} terminated, {r} truncated")
print("lengths of truncated episodes:", sorted({int(n) for n in lengths["R"]}), "(the time limit)")
st = env.episode_stats()
print(f"device-side stats agree: {st['episodes']} episodes finished "
      f"(host log: {len(events)})")

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"nworld": NWORLD, "nstep": NSTEP, "max_steps": MAX_STEPS, "dt": env.dt,
                   "labels": [f"k={s:.2f}" for s in scale], "segments": segments}, f,
                  separators=(",", ":"))
    print("wrote", args.dump_json)

### 6. A baseline first: evolution strategies
NumPy ES with a linear policy, one candidate per world. Saves `policy_es.npz`.

In [ ]:
# Example 30.5 - ex5_es_baseline.py
"""Example 30.5 - CPU-friendly baseline: evolution strategies (NumPy) with a linear policy.

Policy: a = clip(w . obs + b, -1, 1), 6 numbers. Each iteration draws P random perturbations
eps_i and evaluates theta + sigma*eps_i and theta - sigma*eps_i, one candidate per world, so
NWORLD = 2P candidates run in ONE batched rollout. Fitness = return of the first episode
(the time limit equals the rollout horizon). Update: rank-normalized antithetic gradient
estimate (Salimans et al. 2017). Runs on CPU in seconds; saves the policy to policy_es.npz,
which Example 30.7 evaluates when no PPO checkpoint exists.
Run: python ex5_es_baseline.py [--iters N] [--dump-json PATH]
"""
import argparse
import json
import pathlib
import sys

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

HERE = pathlib.Path(__file__).parent if "__file__" in globals() else pathlib.Path.cwd()
sys.path.insert(0, str(HERE))
from ex1_env import CartpoleEnv, OBS_DIM

parser = argparse.ArgumentParser()
parser.add_argument("--iters", type=int, default=None)
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

np.set_printoptions(precision=3, suppress=True)
ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 32           # 2 * population
ITERS = args.iters or (50 if ON_GPU else 30)
HORIZON = 200                             # control steps = 4 s; also the time limit
SIGMA, LR = 0.1, 0.5


def rollout(env, params):
    """Run every world with its own linear policy for HORIZON steps; first-episode returns."""
    w, b = params[:, :OBS_DIM], params[:, OBS_DIM]
    env.reset()
    act = wp.zeros((env.nworld, 1), dtype=float)
    alive = np.ones(env.nworld, dtype=bool)
    ret = np.zeros(env.nworld)
    length = np.zeros(env.nworld, dtype=int)
    for _ in range(HORIZON):
        obs = env.obs_wp.numpy()          # host round trip: fine for a CPU baseline
        act.assign(np.clip((w * obs).sum(1) + b, -1, 1).reshape(-1, 1).astype(np.float32))
        env.step(act)
        ret += alive * env.reward_wp.numpy()
        length += alive
        alive &= ~(env.terminated_wp.numpy() | env.truncated_wp.numpy())
    return ret, length


def ranks(x):
    """Centered ranks in [-0.5, 0.5]: robust to the scale of the returns."""
    r = np.empty(len(x))
    r[np.argsort(x)] = np.arange(len(x))
    return r / (len(x) - 1) - 0.5


env = CartpoleEnv(NWORLD, backend="warp", max_steps=HORIZON, seed=0)
rng = np.random.default_rng(0)
theta = np.zeros(OBS_DIM + 1)
P = NWORLD // 2
history = []
print("device:", wp.get_device(), "| candidates per iteration:", NWORLD, "| params:", theta.size)
for it in range(ITERS):
    eps = rng.standard_normal((P, theta.size))
    cand = np.concatenate([theta + SIGMA * eps, theta - SIGMA * eps])
    env.seed.assign([it])                 # new initial states every iteration
    ret, length = rollout(env, cand)
    f = ranks(ret)
    grad = ((f[:P] - f[P:])[:, None] * eps).sum(0) / (P * SIGMA)
    theta += LR * SIGMA * grad            # step size LR*SIGMA in parameter space
    mean_ret, mean_len = (ret[:P] + ret[P:]).mean() / 2, length.mean()
    history.append([it, round(float(mean_ret), 2), round(float(mean_len), 1)])
    print(f"iter {it:2d}: mean return {mean_ret:7.2f} | mean length {mean_len:5.1f} "
          f"| best {ret.max():7.2f}")

# evaluate the mean policy (no noise) in every world
ret, length = rollout(env, np.tile(theta, (NWORLD, 1)))
print("policy w, b:", theta[:OBS_DIM], round(float(theta[OBS_DIM]), 3))
print(f"final policy: mean return {ret.mean():.2f}, worlds reaching the {HORIZON}-step limit: "
      f"{int((length == HORIZON).sum())}/{NWORLD}")
np.savez("policy_es.npz", w=theta[:OBS_DIM], b=theta[OBS_DIM:])
print("saved policy_es.npz")
if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"source": "ex5_es_baseline.py", "nworld": NWORLD, "horizon": HORIZON,
                   "history": history}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

### 8. Training with the reference PPO
On a GPU this trains 150 iterations of 4096 worlds and saves `policy_ppo.pt` (checkpoint). Watch the mean length climb to the 200-step time limit.

In [ ]:
# Example 30.6 - ex6_ppo_train.py
"""Example 30.6 - train the cartpole balance task with the reference PPO (ppo_reference.py).

GPU budget: 4096 worlds x 32 steps x 150 iterations (about 20M env steps; print your own time).
CPU budget: 3 tiny iterations with 16 worlds, enough to check that everything runs end to end.
Prints iteration, mean return of the last finished episodes, mean length and env-steps/s, and
saves the policy to policy_ppo.pt, which Examples 30.7 and 30.8 load.
Run: python ex6_ppo_train.py [--iters N] [--nworld N] [--dump-json PATH]
"""
import argparse
import json
import pathlib
import sys

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

HERE = pathlib.Path(__file__).parent if "__file__" in globals() else pathlib.Path.cwd()
sys.path.insert(0, str(HERE))

parser = argparse.ArgumentParser()
parser.add_argument("--iters", type=int, default=None)
parser.add_argument("--nworld", type=int, default=None)
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

try:
    import torch
except ImportError:
    torch = None
    print("skipped: torch not installed (PPO needs torch; Example 30.5 trains without it)")

if torch is not None:
    from ex1_env import CartpoleEnv
    from ppo_reference import PPO, PPOConfig

    ON_GPU = wp.get_device().is_cuda
    NWORLD = args.nworld or (4096 if ON_GPU else 16)
    ITERS = args.iters or (150 if ON_GPU else 3)
    env = CartpoleEnv(NWORLD, backend="torch", seed=0)
    agent = PPO(env, PPOConfig(n_steps=32), seed=0)
    print("device:", wp.get_device(), "| torch device:", agent.device, "| nworld:", NWORLD,
          "| iterations:", ITERS)
    print("samples per iteration:", agent.cfg.n_steps * NWORLD,
          "| policy parameters:", sum(p.numel() for p in agent.model.parameters()))
    history = agent.train(ITERS)
    agent.save("policy_ppo.pt")
    print("saved policy_ppo.pt | overflow:",
          "none" if not env.d.overflow.numpy().any() else env.d.overflow.numpy())
    if args.dump_json:
        rows = [[h["iter"], round(h["mean_return"], 2), round(h["mean_length"], 1)]
                for h in history]
        with open(args.dump_json, "w") as f:
            json.dump({"source": "ex6_ppo_train.py", "nworld": NWORLD, "history": rows}, f,
                      separators=(",", ":"))
        print("wrote", args.dump_json)

**CPU fallback.** Without a GPU, the cell above ran only 3 iterations, so its checkpoint is untrained. The next cells write the same file and run it with 40 iterations of 128 worlds (a usable policy; about 10 s on the course's CPU dev box). Skip them on a GPU runtime.

In [ ]:
%%writefile ex6_ppo_train.py
"""Example 30.6 - train the cartpole balance task with the reference PPO (ppo_reference.py).

GPU budget: 4096 worlds x 32 steps x 150 iterations (about 20M env steps; print your own time).
CPU budget: 3 tiny iterations with 16 worlds, enough to check that everything runs end to end.
Prints iteration, mean return of the last finished episodes, mean length and env-steps/s, and
saves the policy to policy_ppo.pt, which Examples 30.7 and 30.8 load.
Run: python ex6_ppo_train.py [--iters N] [--nworld N] [--dump-json PATH]
"""
import argparse
import json
import pathlib
import sys

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

HERE = pathlib.Path(__file__).parent if "__file__" in globals() else pathlib.Path.cwd()
sys.path.insert(0, str(HERE))

parser = argparse.ArgumentParser()
parser.add_argument("--iters", type=int, default=None)
parser.add_argument("--nworld", type=int, default=None)
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

try:
    import torch
except ImportError:
    torch = None
    print("skipped: torch not installed (PPO needs torch; Example 30.5 trains without it)")

if torch is not None:
    from ex1_env import CartpoleEnv
    from ppo_reference import PPO, PPOConfig

    ON_GPU = wp.get_device().is_cuda
    NWORLD = args.nworld or (4096 if ON_GPU else 16)
    ITERS = args.iters or (150 if ON_GPU else 3)
    env = CartpoleEnv(NWORLD, backend="torch", seed=0)
    agent = PPO(env, PPOConfig(n_steps=32), seed=0)
    print("device:", wp.get_device(), "| torch device:", agent.device, "| nworld:", NWORLD,
          "| iterations:", ITERS)
    print("samples per iteration:", agent.cfg.n_steps * NWORLD,
          "| policy parameters:", sum(p.numel() for p in agent.model.parameters()))
    history = agent.train(ITERS)
    agent.save("policy_ppo.pt")
    print("saved policy_ppo.pt | overflow:",
          "none" if not env.d.overflow.numpy().any() else env.d.overflow.numpy())
    if args.dump_json:
        rows = [[h["iter"], round(h["mean_return"], 2), round(h["mean_length"], 1)]
                for h in history]
        with open(args.dump_json, "w") as f:
            json.dump({"source": "ex6_ppo_train.py", "nworld": NWORLD, "history": rows}, f,
                      separators=(",", ":"))
        print("wrote", args.dump_json)

In [ ]:
# CPU fallback: retrain with 40 iterations of 128 worlds (overwrites policy_ppo.pt)
!python ex6_ppo_train.py --iters 40 --nworld 128

### 9. Evaluation under randomized physics
Pole mass from 0.5x to 32x, applied eagerly with `my_env.scale_body_mass` (assign + `mjw.set_const`) between evaluation rollouts. Run it once with the PPO checkpoint (default) and once with `--policy es` to compare. Without a GPU, Example 30.6 runs only 3 iterations and its checkpoint is untrained (0% everywhere): for a usable policy, run the CPU-fallback cell above (40 iterations of 128 worlds) before this cell.

In [ ]:
# Example 30.7 - ex7_evaluate.py
"""Example 30.7 - evaluate the trained policy under randomized pole masses.

Loads policy_ppo.pt (Example 30.6) if it exists and torch is installed, else policy_es.npz
(Example 30.5), and prints which one it used. Mass is a field WITH derived quantities
(body_subtreemass, dof_invweight0, ...), so the pole mass is randomized EAGERLY between
evaluation rollouts with my_env.scale_body_mass (assign + mjw.set_const), never inside the
captured graph. Motor gain and slider damping stay randomized per episode inside the graph.
A rollout succeeds when the pole is still up at the 200-step (4 s) time limit.
Run: python ex7_evaluate.py [--policy ppo|es]
"""
import argparse
import os
import pathlib
import sys

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

HERE = pathlib.Path(__file__).parent if "__file__" in globals() else pathlib.Path.cwd()
sys.path.insert(0, str(HERE))
from ex1_env import CartpoleEnv, my_env

parser = argparse.ArgumentParser()
parser.add_argument("--policy", choices=["ppo", "es"], default=None)
args, _ = parser.parse_known_args()

try:
    import torch
except ImportError:
    torch = None

ON_GPU = wp.get_device().is_cuda
SCALES = np.array([0.5, 2.0, 8.0, 16.0, 32.0])     # pole mass / nominal (0.1 kg)
PER_BIN = 256 if ON_GPU else 8
NWORLD = len(SCALES) * PER_BIN
ROLLOUTS, HORIZON = 2, 200

use_ppo = (args.policy != "es" and torch is not None and os.path.exists("policy_ppo.pt"))
if args.policy == "ppo" and not use_ppo:
    print("policy_ppo.pt or torch missing: falling back to the ES policy")
env = CartpoleEnv(NWORLD, backend="torch" if use_ppo else "warp", max_steps=HORIZON, seed=10)
if use_ppo:
    from ppo_reference import load_policy
    model = load_policy("policy_ppo.pt", device=str(env.obs.device))
    policy = lambda obs: model.act(obs)                       # torch in, torch out
    print("policy: PPO checkpoint policy_ppo.pt")
else:
    p = np.load("policy_es.npz")
    w, b = p["w"], float(p["b"][0])
    act = wp.zeros((NWORLD, 1), dtype=float)

    def policy(obs):
        act.assign(np.clip(obs.numpy() @ w + b, -1, 1).reshape(-1, 1).astype(np.float32))
        return act
    print("policy: ES linear policy policy_es.npz")

pole = mujoco.mj_name2id(env.mjm, mujoco.mjtObj.mjOBJ_BODY, "pole")
cart = mujoco.mj_name2id(env.mjm, mujoco.mjtObj.mjOBJ_BODY, "cart")
rng = np.random.default_rng(0)
success = np.zeros(len(SCALES))
print("device:", wp.get_device(), "| worlds:", NWORLD, f"({PER_BIN} per mass bin) | rollouts:",
      ROLLOUTS)
for k in range(ROLLOUTS):
    scale = np.repeat(SCALES, PER_BIN) * rng.uniform(0.9, 1.1, NWORLD)   # jitter inside a bin
    my_env.scale_body_mass(env.mjm, env.m, env.d, pole, scale)   # eager DR + set_const
    sub = env.m.body_subtreemass.numpy()[:, cart]
    print(f"rollout {k}: cart subtree mass (cart + pole) min {sub.min():.3f} max {sub.max():.3f}")
    env.seed.assign([100 + k])
    obs = env.reset()
    alive = np.ones(NWORLD, dtype=bool)
    for _ in range(HORIZON):
        obs, rew, term, trunc = env.step(policy(obs))
        t_np = term.cpu().numpy() if use_ppo else term.numpy()
        alive &= ~t_np
    success += alive.reshape(len(SCALES), PER_BIN).mean(1) / ROLLOUTS
for s, rate in zip(SCALES, success):
    print(f"  pole mass x{s:4.1f} ({0.1 * s:4.2f} kg): success {100 * rate:5.1f}% "
          + "#" * int(round(20 * rate)))
print("overflow:", my_env.describe_overflow(env.d))

The same evaluation with the ES policy (the file is written first so it can run as a script):

In [ ]:
%%writefile ex7_evaluate.py
"""Example 30.7 - evaluate the trained policy under randomized pole masses.

Loads policy_ppo.pt (Example 30.6) if it exists and torch is installed, else policy_es.npz
(Example 30.5), and prints which one it used. Mass is a field WITH derived quantities
(body_subtreemass, dof_invweight0, ...), so the pole mass is randomized EAGERLY between
evaluation rollouts with my_env.scale_body_mass (assign + mjw.set_const), never inside the
captured graph. Motor gain and slider damping stay randomized per episode inside the graph.
A rollout succeeds when the pole is still up at the 200-step (4 s) time limit.
Run: python ex7_evaluate.py [--policy ppo|es]
"""
import argparse
import os
import pathlib
import sys

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

HERE = pathlib.Path(__file__).parent if "__file__" in globals() else pathlib.Path.cwd()
sys.path.insert(0, str(HERE))
from ex1_env import CartpoleEnv, my_env

parser = argparse.ArgumentParser()
parser.add_argument("--policy", choices=["ppo", "es"], default=None)
args, _ = parser.parse_known_args()

try:
    import torch
except ImportError:
    torch = None

ON_GPU = wp.get_device().is_cuda
SCALES = np.array([0.5, 2.0, 8.0, 16.0, 32.0])     # pole mass / nominal (0.1 kg)
PER_BIN = 256 if ON_GPU else 8
NWORLD = len(SCALES) * PER_BIN
ROLLOUTS, HORIZON = 2, 200

use_ppo = (args.policy != "es" and torch is not None and os.path.exists("policy_ppo.pt"))
if args.policy == "ppo" and not use_ppo:
    print("policy_ppo.pt or torch missing: falling back to the ES policy")
env = CartpoleEnv(NWORLD, backend="torch" if use_ppo else "warp", max_steps=HORIZON, seed=10)
if use_ppo:
    from ppo_reference import load_policy
    model = load_policy("policy_ppo.pt", device=str(env.obs.device))
    policy = lambda obs: model.act(obs)                       # torch in, torch out
    print("policy: PPO checkpoint policy_ppo.pt")
else:
    p = np.load("policy_es.npz")
    w, b = p["w"], float(p["b"][0])
    act = wp.zeros((NWORLD, 1), dtype=float)

    def policy(obs):
        act.assign(np.clip(obs.numpy() @ w + b, -1, 1).reshape(-1, 1).astype(np.float32))
        return act
    print("policy: ES linear policy policy_es.npz")

pole = mujoco.mj_name2id(env.mjm, mujoco.mjtObj.mjOBJ_BODY, "pole")
cart = mujoco.mj_name2id(env.mjm, mujoco.mjtObj.mjOBJ_BODY, "cart")
rng = np.random.default_rng(0)
success = np.zeros(len(SCALES))
print("device:", wp.get_device(), "| worlds:", NWORLD, f"({PER_BIN} per mass bin) | rollouts:",
      ROLLOUTS)
for k in range(ROLLOUTS):
    scale = np.repeat(SCALES, PER_BIN) * rng.uniform(0.9, 1.1, NWORLD)   # jitter inside a bin
    my_env.scale_body_mass(env.mjm, env.m, env.d, pole, scale)   # eager DR + set_const
    sub = env.m.body_subtreemass.numpy()[:, cart]
    print(f"rollout {k}: cart subtree mass (cart + pole) min {sub.min():.3f} max {sub.max():.3f}")
    env.seed.assign([100 + k])
    obs = env.reset()
    alive = np.ones(NWORLD, dtype=bool)
    for _ in range(HORIZON):
        obs, rew, term, trunc = env.step(policy(obs))
        t_np = term.cpu().numpy() if use_ppo else term.numpy()
        alive &= ~t_np
    success += alive.reshape(len(SCALES), PER_BIN).mean(1) / ROLLOUTS
for s, rate in zip(SCALES, success):
    print(f"  pole mass x{s:4.1f} ({0.1 * s:4.2f} kg): success {100 * rate:5.1f}% "
          + "#" * int(round(20 * rate)))
print("overflow:", my_env.describe_overflow(env.d))

In [ ]:
!python ex7_evaluate.py --policy es

Render the policy: `mujoco.Renderer` (EGL on Colab) for world 0 as a video, and the MJWarp batch renderer for a 4x4 grid of all worlds.

In [ ]:
# Example 30.8 - ex8_render_policy.py
"""Example 30.8 - render the trained policy: world 0 as frames and a 4x4 batch-render grid.

16 worlds with 16 different pole masses (eager mass DR + set_const) run the PPO policy if
policy_ppo.pt exists (and torch is installed), else the ES policy. mujoco.Renderer (C MuJoCo,
OpenGL) renders world 0 when an OpenGL backend exists; otherwise the MJWarp batch renderer
renders ALL worlds in one launch (works on Warp's CPU device), which also gives the 4x4 grid.
Run: python ex8_render_policy.py [--policy ppo|es] [--dump-json PATH] [--dump-png DIR]
  --dump-json PATH  decimated (cart x, pole angle) per world for the page's replay player
  --dump-png DIR    ex8_grid.png (4x4 batch render, last frame) and ex8_world0.png (a strip)
"""
import argparse
import json
import os
import pathlib
import struct
import sys
import zlib

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

HERE = pathlib.Path(__file__).parent if "__file__" in globals() else pathlib.Path.cwd()
sys.path.insert(0, str(HERE))
from ex1_env import CartpoleEnv, my_env

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
parser.add_argument("--dump-png", default=None)
parser.add_argument("--policy", choices=["ppo", "es"], default=None)
args, _ = parser.parse_known_args()

try:
    import torch
except ImportError:
    torch = None

ON_GPU = wp.get_device().is_cuda
NWORLD, HORIZON, EVERY = 16, 200, 4
W, H = (128, 128) if ON_GPU else (64, 64)


def save_png(path, img):
    """Write an (H, W, 3) uint8 array as an RGB PNG (standard library only)."""
    h, w, _ = img.shape
    raw = b"".join(b"\x00" + img[y].tobytes() for y in range(h))

    def chunk(tag, data):
        return struct.pack(">I", len(data)) + tag + data + struct.pack(">I", zlib.crc32(tag + data))

    with open(path, "wb") as f:
        f.write(b"\x89PNG\r\n\x1a\n" + chunk(b"IHDR", struct.pack(">IIBBBBB", w, h, 8, 2, 0, 0, 0))
                + chunk(b"IDAT", zlib.compress(raw, 9)) + chunk(b"IEND", b""))


use_ppo = args.policy != "es" and torch is not None and os.path.exists("policy_ppo.pt")
env = CartpoleEnv(NWORLD, backend="torch" if use_ppo else "warp", max_steps=HORIZON, seed=21)
if use_ppo:
    from ppo_reference import load_policy
    model = load_policy("policy_ppo.pt", device=str(env.obs.device))
    policy = lambda obs: model.act(obs)
    print("policy: PPO checkpoint policy_ppo.pt")
else:
    p = np.load("policy_es.npz")
    w, b = p["w"], float(p["b"][0])
    act = wp.zeros((NWORLD, 1), dtype=float)

    def policy(obs):
        act.assign(np.clip(obs.numpy() @ w + b, -1, 1).reshape(-1, 1).astype(np.float32))
        return act
    print("policy: ES linear policy policy_es.npz")

scale = np.geomspace(0.5, 32.0, NWORLD)                   # one pole mass per world
pole = mujoco.mj_name2id(env.mjm, mujoco.mjtObj.mjOBJ_BODY, "pole")
my_env.scale_body_mass(env.mjm, env.m, env.d, pole, scale)

# renderer: C MuJoCo for world 0 if OpenGL works, else the MJWarp batch renderer for all worlds
renderer, rc = None, None
try:
    renderer = mujoco.Renderer(env.mjm, H, W)
    mjd = mujoco.MjData(env.mjm)
    print("renderer: mujoco.Renderer (OpenGL)")
except Exception as e:
    print(f"mujoco.Renderer unavailable ({type(e).__name__}); using the MJWarp batch renderer")
rc = mjw.create_render_context(env.mjm, nworld=NWORLD, cam_res=(W, H), render_rgb=True)
rgb = wp.zeros((NWORLD, H, W), dtype=wp.vec3)


def render_batch():
    mjw.forward(env.m, env.d)       # poses for the current qpos (reset worlds included)
    mjw.refit_bvh(env.m, env.d, rc)
    mjw.render(env.m, env.d, rc)
    mjw.get_rgb(rc, 0, rgb)
    return (np.clip(rgb.numpy(), 0, 1) * 255).astype(np.uint8)


obs = env.reset()
frames, video, alive = [], [], np.ones(NWORLD, dtype=bool)
for i in range(HORIZON):
    obs, rew, term, trunc = env.step(policy(obs))
    alive &= ~(term.cpu().numpy() if use_ppo else term.numpy())
    if i % EVERY == 0:
        q = env.d.qpos.numpy()
        frames.append([[round(float(x), 3), round(float(t), 3)] for x, t in q])
        if renderer is not None:
            mjw.get_data_into(mjd, env.mjm, env.d, world_id=0)
            renderer.update_scene(mjd, camera="front")
            video.append(renderer.render())
        elif i % (4 * EVERY) == 0:
            video.append(render_batch()[0])
grid = render_batch()
print("device:", wp.get_device(), f"| {NWORLD} worlds | {HORIZON} steps | frames kept "
      f"{len(frames)} | video frames {len(video)} of {video[0].shape}")
print("still balancing at the end:", int(alive.sum()), "/", NWORLD,
      "| pole mass factor of the failures:", np.round(scale[~alive], 2).tolist())
print("batch render:", grid.shape, grid.dtype, "| mean RGB world 0:",
      grid[0].reshape(-1, 3).mean(0).round(1))
try:
    import mediapy as media
    media.write_video("ex8_world0.mp4", video, fps=1 / (EVERY * env.dt))
    print("wrote ex8_world0.mp4")
except ImportError:
    print("skipped: mediapy not installed (no video file)")

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"nworld": NWORLD, "dt": EVERY * env.dt, "kind": "cartpole",
                   "labels": [f"m x{s:.2f}" for s in scale], "alive": alive.tolist(),
                   "frames": frames}, f, separators=(",", ":"))
    print("wrote", args.dump_json, len(frames), "frames")
if args.dump_png:
    os.makedirs(args.dump_png, exist_ok=True)
    tiles = grid.reshape(4, 4, H, W, 3).transpose(0, 2, 1, 3, 4).reshape(4 * H, 4 * W, 3)
    save_png(os.path.join(args.dump_png, "ex8_grid.png"), tiles)
    strip = np.concatenate(video[:8], axis=1)
    save_png(os.path.join(args.dump_png, "ex8_world0.png"), strip)
    print("wrote", tiles.shape, "grid and", strip.shape, "strip to", args.dump_png)

In [ ]:
try:
    import mediapy as media
    media.show_video(media.read_video("ex8_world0.mp4"), fps=12)
except Exception as e:
    print("skipped video:", type(e).__name__, e)

## Contact track
### 10. HumanoidStandEnv
Step 1: measure contacts, rows and broadphase candidates with random actions, add 25%, and sweep the solver.

In [ ]:
# Example 30.9 - ex9_humanoid_sizing.py
"""Example 30.9 - contact track, step 1: size the buffers and pick the solver by measuring.

The Ch 9 recipe: run the real task with GENEROUS budgets and a policy that visits the bad states
(random actions make the humanoid fall, so feet, knees, hands and torso all hit the floor), and
record three worst cases: contacts per world, constraint rows per world, and the broadphase
candidate count d.ncollision (a TOTAL over all worlds, checked against naconmax). Add a 25%
margin. Then a small solver sweep on the same rollout: Newton vs CG, pyramidal vs elliptic cone,
3 vs 10 iterations. cone and solver are compile-time choices (new kernels per variant);
iterations is a runtime value. CPU numbers are from Warp's CPU device ("on a CPU dev box"),
not GPU performance.
Run: python ex9_humanoid_sizing.py [--dump-json PATH]
"""
import argparse
import json
import math
import time

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from etils import epath

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 1024 if ON_GPU else 4
NSTEP = 1000 if ON_GPU else 120          # 0.005 s per step: 5 s on GPU, 0.6 s on CPU
HUMANOID_PATH = epath.resource_path("mujoco_warp") / "test_data/humanoid/humanoid.xml"


def humanoid_model():
    """The shipped humanoid (nq 28, nv 27, nu 21) plus three torso sensors added with MjSpec."""
    spec = mujoco.MjSpec.from_file(str(HUMANOID_PATH))
    xbody, body = mujoco.mjtObj.mjOBJ_XBODY, mujoco.mjtObj.mjOBJ_BODY   # XBODY = body frame;
    for name, kind, obj in [("torso_pos", mujoco.mjtSensor.mjSENS_FRAMEPOS, xbody),  # BODY would
                            ("torso_up", mujoco.mjtSensor.mjSENS_FRAMEZAXIS, xbody),  # be the COM
                            ("com_vel", mujoco.mjtSensor.mjSENS_SUBTREELINVEL, body)]:
        spec.add_sensor(name=name, type=kind, objtype=obj, objname="torso")
    return spec.compile()


def rollout(mjm, naconmax, njmax, iterations, measure):
    """Random actions from qpos0 + noise in every world; returns stats (and steps/s if timed)."""
    m = mjw.put_model(mjm)
    m.opt.iterations = iterations                 # runtime value: no recompile
    m.opt.warn_overflow = 0                       # no device printf; the bits are still set
    d = mjw.make_data(mjm, nworld=NWORLD, naconmax=naconmax, njmax=njmax)
    rng = np.random.default_rng(0)                # same states and actions for every variant
    qpos = np.tile(mjm.qpos0, (NWORLD, 1))
    qpos[:, 7:] += rng.uniform(-0.05, 0.05, (NWORLD, mjm.nq - 7))
    d.qpos.assign(qpos.astype(np.float32))
    acts = rng.uniform(-1, 1, (NSTEP, NWORLD, mjm.nu)).astype(np.float32)
    ctrl = [wp.array(a) for a in acts]
    mjw.step(m, d)                                # warm-up: compile this variant
    max_con = np.zeros(NWORLD, dtype=int)
    max_efc = np.zeros(NWORLD, dtype=int)
    max_ncol = 0
    niter, iters_bit = [], 0
    wp.synchronize()
    t0 = time.perf_counter()
    for i in range(NSTEP):
        wp.copy(d.ctrl, ctrl[i])
        mjw.step(m, d)
        if measure:                               # host reads: measurement only, never in RL
            n = int(d.nacon.numpy()[0])
            wid = d.contact.worldid.numpy()[:min(n, d.naconmax)]
            max_con = np.maximum(max_con, np.bincount(wid, minlength=NWORLD))
            max_efc = np.maximum(max_efc, d.nefc.numpy())
            max_ncol = max(max_ncol, int(d.ncollision.numpy()[0]))
            niter.append(d.solver_niter.numpy().mean())
    wp.synchronize()
    sec = time.perf_counter() - t0
    ov = d.overflow.numpy()
    iters_bit = int(np.count_nonzero(ov & mjw.OverflowType.ITERATIONS))
    other = int(np.count_nonzero(ov & ~int(mjw.OverflowType.ITERATIONS)))
    bits = sorted({b.name for b in mjw.OverflowType if b != mjw.OverflowType.ITERATIONS
                   and (ov & b).any()})
    return dict(max_con=max_con, max_efc=max_efc, max_ncol=max_ncol,
                niter=float(np.mean(niter)) if niter else 0.0,
                iters_bit=iters_bit, other_overflow=other, bits=bits, sps=NSTEP * NWORLD / sec,
                z=d.qpos.numpy()[:, 2])


mjm = humanoid_model()
print("device:", wp.get_device(), "| nworld:", NWORLD, "| steps:", NSTEP,
      f"| nv {mjm.nv}, nu {mjm.nu}, ngeom {mjm.ngeom}")
# 1. measurement rollout with generous budgets (default solver: Newton, pyramidal, 10 iters)
big = rollout(mjm, naconmax=64 * NWORLD, njmax=512, iterations=10, measure=True)
assert big["other_overflow"] == 0, "the measurement budget itself overflowed: raise it"
con, efc, ncol = int(big["max_con"].max()), int(big["max_efc"].max()), big["max_ncol"]
NACONMAX = math.ceil(1.25 * max(ncol, int(big["max_con"].sum())))   # the pool holds BOTH
NJMAX = math.ceil(1.25 * efc)
print(f"torso height at the end: {np.round(big['z'], 2)} (standing starts at 1.28)")
print(f"max contacts per world: {big['max_con'].tolist()} -> worst {con}")
print(f"max constraint rows per world: {big['max_efc'].tolist()} -> worst {efc}")
print(f"max broadphase candidates (all worlds): {ncol} = {ncol / NWORLD:.1f} per world, "
      f"vs {con} contacts in the worst world")
print(f"recommended: naconmax={NACONMAX} ({NACONMAX / NWORLD:.1f} per world), njmax={NJMAX} "
      f"(worst case + 25%)")

# 2. solver sweep on the same rollout with the recommended sizes
rows = []
print(f"{'solver':7s} {'cone':10s} {'iters':>5s} {'mean niter':>10s} {'ITER bit':>8s} "
      f"{'max rows':>8s} {'steps/s':>9s}  other overflow")
for solver in (mujoco.mjtSolver.mjSOL_NEWTON, mujoco.mjtSolver.mjSOL_CG):
    for cone in (mujoco.mjtCone.mjCONE_PYRAMIDAL, mujoco.mjtCone.mjCONE_ELLIPTIC):
        for iters in (3, 10):
            mjm.opt.solver, mjm.opt.cone = solver, cone        # compile-time choices
            mes = rollout(mjm, NACONMAX, NJMAX, iters, measure=True)
            tim = rollout(mjm, NACONMAX, NJMAX, iters, measure=False)
            row = [solver.name[6:], cone.name[7:].lower(), iters, round(mes["niter"], 2),
                   mes["iters_bit"], int(mes["max_efc"].max()), round(tim["sps"]),
                   "|".join(mes["bits"]) or "none"]
            rows.append(row)
            print(f"{row[0]:7s} {row[1]:10s} {iters:5d} {row[3]:10.2f} {row[4]:5d}/{NWORLD} "
                  f"{row[5]:8d} {row[6]:9.0f}  {row[7]}")
print("ITER bit = worlds whose solver hit the iteration cap at least once (sticky until reset)")
if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"source": "ex9_humanoid_sizing.py", "nworld": NWORLD, "nstep": NSTEP,
                   "max_con": big["max_con"].tolist(), "max_efc": big["max_efc"].tolist(),
                   "max_ncollision": ncol, "naconmax": NACONMAX, "njmax": NJMAX, "sweep": rows},
                  f, separators=(",", ":"))
    print("wrote", args.dump_json)

Step 2: the environment (sizes from step 1; overflow monitoring that survives auto-reset; observations from qpos, not sensordata; a per-rollout contact budget). The cell below runs the file the `%%writefile ex10_humanoid_env.py` cell wrote. If your GPU run of 30.9 printed different worst cases, edit `NACON_PER_WORLD` and `NJMAX` in that `%%writefile` cell, re-run it, then re-run the cell below.

In [ ]:
# Example 30.10 - ex10_humanoid_env.py (runs the file written by the %%writefile cell above)
!python ex10_humanoid_env.py

Step 3: train (GPU budget 20-30 minutes). The overflow column decodes every bit any world raised during that rollout; the contact column shows the worst `nacon` and `nefc` against their budgets and the mean solver iterations.

In [ ]:
# If you edited and re-wrote ex10_humanoid_env.py, load the new version before training.
import importlib, sys
if "ex10_humanoid_env" in sys.modules:
    importlib.reload(sys.modules["ex10_humanoid_env"])
    print("reloaded ex10_humanoid_env")

In [ ]:
# Example 30.11 - ex11_humanoid_train.py
"""Example 30.11 - contact track, step 3: train HumanoidStandEnv with the reference PPO.

GPU budget: 2048 worlds x 32 steps x 300 iterations (about 20M env steps, 80M physics steps;
print your own time, about 20-30 min on a Colab GPU is the target). CPU budget: 2 tiny iterations
with 8 worlds, to check the pipeline end to end. After every iteration it prints the mean
episode length and return of the last finished episodes and the overflow bits seen during the
rollout (decoded from env.overflow_seen, which survives auto-resets), plus the contact budget of
that rollout from env.contact_stats(): worst nacon against naconmax, worst nefc against njmax and
the mean solver_niter. Saves policy_humanoid.pt.
Run: python ex11_humanoid_train.py [--iters N] [--nworld N]
"""
import argparse
import pathlib
import sys

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

HERE = pathlib.Path(__file__).parent if "__file__" in globals() else pathlib.Path.cwd()
sys.path.insert(0, str(HERE))

parser = argparse.ArgumentParser()
parser.add_argument("--iters", type=int, default=None)
parser.add_argument("--nworld", type=int, default=None)
args, _ = parser.parse_known_args()

try:
    import torch
except ImportError:
    torch = None
    print("skipped: torch not installed (PPO needs torch)")

if torch is not None:
    from ex10_humanoid_env import HumanoidStandEnv
    from ppo_reference import PPO, PPOConfig

    ON_GPU = wp.get_device().is_cuda
    NWORLD = args.nworld or (2048 if ON_GPU else 8)
    ITERS = args.iters or (300 if ON_GPU else 2)
    env = HumanoidStandEnv(NWORLD, backend="torch", seed=0)
    agent = PPO(env, PPOConfig(n_steps=32, hidden=(128, 128), lr=3e-4), seed=0)
    print("device:", wp.get_device(), "| nworld:", NWORLD, "| iterations:", ITERS,
          f"| obs {agent.obs_dim}, actions {agent.act_dim}")
    for it in range(ITERS):
        h = agent.train(1, log=None)[0]
        ov = env.overflow()                      # bits raised during this rollout, then cleared
        cs = env.contact_stats()                 # worst nacon / nefc, mean niter, then cleared
        print(f"iter {it:3d} | mean length {h['mean_length']:6.1f} | mean return "
              f"{h['mean_return']:8.2f} | {h['steps_per_s']:8.0f} env-steps/s | {cs['text']} | "
              "overflow: " + ov.replace("\n", "; "))
    agent.save("policy_humanoid.pt")
    print("saved policy_humanoid.pt")

## 11. Stretch sections (optional)
<details><summary>Scaling study and swing-up</summary>

Example 30.12 keeps the whole ES rollout on the device (the policy is a Warp kernel) and compares nworld values under a fixed wall-clock budget. Example 30.13 is the swing-up variant with an energy-shaped reward.
</details>

They do not run by default: "Run all" only writes the two files. Set `RUN_STRETCH = True` in the cell after them to run both (Example 30.12 alone takes up to 60 s per nworld row on a GPU).

In [ ]:
%%writefile ex12_scaling.py
"""Example 30.12 (stretch) - scaling study: more worlds per update under a fixed wall-clock budget.

Evolution strategies on the balance task, but now the WHOLE rollout stays on the device: the
linear policy of every candidate is a Warp kernel that reads env.obs_wp and writes env.action,
and a second kernel adds the reward of each world's first episode. The host only touches the
data once per iteration (to rank the candidates). For each nworld we run ES for the same
wall-clock budget and report env-steps/s and the return reached. Bigger populations give
better gradient estimates per iteration; on a GPU they also cost almost nothing extra.
CPU numbers are from Warp's CPU device ("on a CPU dev box"), NOT GPU performance.
Run: python ex12_scaling.py [--budget SECONDS]
"""
import argparse
import pathlib
import sys
import time

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

HERE = pathlib.Path(__file__).parent if "__file__" in globals() else pathlib.Path.cwd()
sys.path.insert(0, str(HERE))
from ex1_env import CartpoleEnv, OBS_DIM

parser = argparse.ArgumentParser()
parser.add_argument("--budget", type=float, default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
SIZES = [256, 1024, 4096, 16384] if ON_GPU else [16, 64, 256]
BUDGET = args.budget or (60.0 if ON_GPU else 4.0)        # seconds per nworld
HORIZON, SIGMA, LR = 200, 0.1, 0.5


@wp.kernel
def linear_policy(obs: wp.array2d(dtype=float), params: wp.array2d(dtype=float),
                  action: wp.array2d(dtype=float)):
    worldid = wp.tid()
    a = params[worldid, 5]                              # bias
    for i in range(5):
        a += params[worldid, i] * obs[worldid, i]
    action[worldid, 0] = wp.clamp(a, -1.0, 1.0)


@wp.kernel
def first_episode(reward: wp.array(dtype=float), done: wp.array(dtype=bool),
                  alive: wp.array(dtype=bool), ret: wp.array(dtype=float)):
    worldid = wp.tid()
    if alive[worldid]:
        ret[worldid] = ret[worldid] + reward[worldid]
        if done[worldid]:
            alive[worldid] = False


def es_run(nworld, budget):
    env = CartpoleEnv(nworld, backend="warp", max_steps=HORIZON, seed=0)
    env.reset()
    params = wp.zeros((nworld, OBS_DIM + 1), dtype=float)
    alive = wp.zeros(nworld, dtype=bool)
    ret = wp.zeros(nworld, dtype=float)
    rng = np.random.default_rng(0)
    theta, P, it, steps = np.zeros(OBS_DIM + 1), nworld // 2, 0, 0
    best = []
    t0 = time.perf_counter()
    while it == 0 or time.perf_counter() - t0 < budget:
        eps = rng.standard_normal((P, theta.size))
        params.assign(np.concatenate([theta + SIGMA * eps, theta - SIGMA * eps]).astype(np.float32))
        env.seed.assign([it])
        env.reset()
        alive.fill_(True)
        ret.zero_()
        for _ in range(HORIZON):                        # no host sync inside this loop
            wp.launch(linear_policy, dim=nworld, inputs=[env.obs_wp, params], outputs=[env.action])
            wp.capture_launch(env.graph)
            wp.launch(first_episode, dim=nworld, inputs=[env.reward_wp, env.done],
                      outputs=[alive, ret])
        r = ret.numpy()                                 # ONE sync per iteration
        rank = np.empty(nworld)
        rank[np.argsort(r)] = np.arange(nworld)
        f = rank / (nworld - 1) - 0.5
        theta += LR * ((f[:P] - f[P:])[:, None] * eps).sum(0) / P
        best.append(float(r.mean()))
        it += 1
        steps += nworld * HORIZON
    sec = time.perf_counter() - t0
    return steps / sec, it, best[-1]


print("device:", wp.get_device(), f"| budget {BUDGET:.0f} s per row | horizon {HORIZON}")
print(f"{'nworld':>7s} {'env-steps/s':>12s} {'iterations':>10s} {'mean return (last iter)':>24s}")
for n in SIZES:
    es_run(n, 0.0)                                      # warm-up: compile + capture outside
    sps, iters, last = es_run(n, BUDGET)
    print(f"{n:7d} {sps:12.0f} {iters:10d} {last:24.1f}")
print("the maximum return is about 200 (1 per step for 200 steps)")

In [ ]:
%%writefile ex13_swingup.py
"""Example 30.13 (stretch) - swing-up: start hanging down (hinge = pi), energy-shaped reward.

CartpoleEnv(swingup=True) starts every episode at theta = pi + noise, never terminates on the
angle, and rewards 0.5*(1 + cos theta) plus 0.5*exp(-2*|E - E_up|), a bonus for having the
pole energy E of the upright pole (energy shaping), minus small cart-position and effort costs.
A linear policy cannot swing up AND balance, so this ES baseline uses a few nonlinear features:
[obs, theta_dot*cos(theta), sin(theta)*cos(theta)]. Success = the pole is within 20 degrees of upright at the end.
CPU budget: 32 candidates, 25 iterations; GPU: 4096 candidates, 100 iterations.
Run: python ex13_swingup.py
"""
import pathlib
import sys

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

HERE = pathlib.Path(__file__).parent if "__file__" in globals() else pathlib.Path.cwd()
sys.path.insert(0, str(HERE))
from ex1_env import CartpoleEnv

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 32
ITERS = 100 if ON_GPU else 25
HORIZON, SIGMA, LR = 300, 0.2, 0.5          # 6 s episodes


def features(o):
    return np.concatenate([o, (o[:, 4] * o[:, 1])[:, None], (o[:, 2] * o[:, 1])[:, None],
                           np.ones((len(o), 1))], 1)


def rollout(env, params):
    env.reset()
    act = wp.zeros((env.nworld, 1), dtype=float)
    ret = np.zeros(env.nworld)
    alive = np.ones(env.nworld, dtype=bool)
    for _ in range(HORIZON):
        o = env.obs_wp.numpy()
        act.assign(np.clip((params * features(o)).sum(1), -1, 1).reshape(-1, 1).astype(np.float32))
        env.step(act)
        ret += alive * env.reward_wp.numpy()
        alive &= ~(env.terminated_wp.numpy() | env.truncated_wp.numpy())
    cos_end = env.final_obs_wp.numpy()[:, 1]       # the state the time limit cut
    return ret, cos_end


env = CartpoleEnv(NWORLD, backend="warp", max_steps=HORIZON, swingup=True, seed=0)
rng = np.random.default_rng(1)
theta = np.zeros(8)
P = NWORLD // 2
print("device:", wp.get_device(), "| candidates:", NWORLD, "| iterations:", ITERS)
for it in range(ITERS):
    eps = rng.standard_normal((P, theta.size))
    env.seed.assign([it])
    ret, _ = rollout(env, np.concatenate([theta + SIGMA * eps, theta - SIGMA * eps]))
    rank = np.empty(NWORLD)
    rank[np.argsort(ret)] = np.arange(NWORLD)
    f = rank / (NWORLD - 1) - 0.5
    theta += LR * SIGMA * ((f[:P] - f[P:])[:, None] * eps).sum(0) / (P * SIGMA)
    if it % 5 == 0 or it == ITERS - 1:
        print(f"iter {it:3d}: mean return {ret.mean():7.1f} | best {ret.max():7.1f}")
env.seed.assign([999])
ret, cos_end = rollout(env, np.tile(theta, (NWORLD, 1)))
up = cos_end > np.cos(np.deg2rad(20))
print(f"final policy: mean return {ret.mean():.1f} | success (within 20 deg of upright at "
      f"{HORIZON * env.dt:.0f} s): {int(up.sum())}/{NWORLD} = {100 * up.mean():.0f}%")
print("a linear-in-features policy is a weak swing-up controller: try PPO (ex6) with swingup=True")

In [ ]:
RUN_STRETCH = False   # set to True to run Examples 30.12 and 30.13
import subprocess, sys
if RUN_STRETCH:
    for f in ("ex12_scaling.py", "ex13_swingup.py"):
        print(f"--- {f}", flush=True)
        r = subprocess.run([sys.executable, f], capture_output=True, text=True)
        print(r.stdout, r.stderr)
else:
    print("stretch examples skipped (RUN_STRETCH = False)")

## Exercise 30.1: an action-rate penalty kernel
Subtract `k * sum((a - a_prev)**2)` from each world's reward and store the current action in `prev_action`. Check against NumPy.

In [ ]:
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING

@wp.kernel
def action_rate_penalty(action: wp.array2d(dtype=float), prev_action: wp.array2d(dtype=float),
                        k: float, reward: wp.array(dtype=float)):
    worldid = wp.tid()
    # your code here

N, A, K = 64, 3, 0.05
rng = np.random.default_rng(0)
a_prev = rng.uniform(-1, 1, (N, A)).astype(np.float32)
a = rng.uniform(-1, 1, (N, A)).astype(np.float32)
r0 = rng.normal(size=N).astype(np.float32)
action, prev, reward = wp.array(a), wp.array(a_prev), wp.array(r0)
wp.launch(action_rate_penalty, dim=N, inputs=[action, prev, K], outputs=[reward])
expected = r0 - K * ((a - a_prev) ** 2).sum(1)
assert np.allclose(reward.numpy(), expected, atol=1e-6)
assert np.array_equal(prev.numpy(), a), "prev_action must now hold this step's action"
print("action-rate penalty matches NumPy")

In [ ]:
#@title Exercise 30.1: solution (double-click to show the code)

@wp.kernel
def action_rate_penalty(action: wp.array2d(dtype=float), prev_action: wp.array2d(dtype=float),
                        k: float, reward: wp.array(dtype=float)):
    worldid = wp.tid()
    s = float(0.0)
    for i in range(action.shape[1]):
        da = action[worldid, i] - prev_action[worldid, i]
        s += da * da
        prev_action[worldid, i] = action[worldid, i]   # remember for the next step
    reward[worldid] = reward[worldid] - k * s

N, A, K = 64, 3, 0.05
rng = np.random.default_rng(0)
a_prev = rng.uniform(-1, 1, (N, A)).astype(np.float32)
a = rng.uniform(-1, 1, (N, A)).astype(np.float32)
r0 = rng.normal(size=N).astype(np.float32)
action, prev, reward = wp.array(a), wp.array(a_prev), wp.array(r0)
wp.launch(action_rate_penalty, dim=N, inputs=[action, prev, K], outputs=[reward])
expected = r0 - K * ((a - a_prev) ** 2).sum(1)
assert np.allclose(reward.numpy(), expected, atol=1e-6)
assert np.array_equal(prev.numpy(), a)
print("action-rate penalty matches NumPy")

## Exercise 30.2: where each kind of DR acts
Assert that the in-graph DR changes gain and damping only in worlds reset during that step, and that eager mass DR changes `body_subtreemass` per world.

In [ ]:
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from ex1_env import CartpoleEnv, my_env     # the %%writefile cells provide both files

env = CartpoleEnv(8, backend="warp", seed=5)
env.reset()
act = wp.array(np.full((8, 1), 1.0, np.float32))
resets = 0
for _ in range(100):
    before = env.m.actuator_gainprm.numpy()[:, 0, 0].copy()
    env.step(act)
    done = env.done.numpy()
    after = env.m.actuator_gainprm.numpy()[:, 0, 0]
    # your asserts: running worlds unchanged, reset worlds changed (gain and damping)
    resets += int(done.sum())
assert resets > 0
# your code: scale the pole mass by np.linspace(1, 4, 8) and assert the cart's subtree mass
# equals 1.0 + 0.1 * scale in every world

In [ ]:
#@title Exercise 30.2: solution (double-click to show the code)

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from ex1_env import CartpoleEnv, my_env

env = CartpoleEnv(8, backend="warp", seed=5)
env.reset()
act = wp.array(np.full((8, 1), 1.0, np.float32))      # full push: episodes end quickly
resets = 0
for _ in range(100):
    before = env.m.actuator_gainprm.numpy()[:, 0, 0].copy()
    damp_before = env.m.dof_damping.numpy()[:, 0].copy()
    env.step(act)
    done = env.done.numpy()
    after = env.m.actuator_gainprm.numpy()[:, 0, 0]
    damp_after = env.m.dof_damping.numpy()[:, 0]
    assert np.array_equal(after[~done], before[~done]), "running worlds must keep their gain"
    assert np.array_equal(damp_after[~done], damp_before[~done])
    assert (after[done] != before[done]).all(), "a reset world draws a new gain"
    resets += int(done.sum())
assert resets > 0
pole = mujoco.mj_name2id(env.mjm, mujoco.mjtObj.mjOBJ_BODY, "pole")
cart = mujoco.mj_name2id(env.mjm, mujoco.mjtObj.mjOBJ_BODY, "cart")
my_env.scale_body_mass(env.mjm, env.m, env.d, pole, np.linspace(1, 4, 8))
sub = env.m.body_subtreemass.numpy()[:, cart]
assert np.allclose(sub, 1.0 + 0.1 * np.linspace(1, 4, 8), atol=1e-5), sub
print(f"in-graph DR touched only reset worlds ({resets} resets); subtree mass {sub[:3]}")

## Stretch: learn to stand from depth images
Add 64x64 depth observations to `HumanoidStandEnv` (render context with `render_depth=True`; `forward` + `refit_bvh` + `render` + `get_depth` inside the graph after the reset), give PPO a small CNN, and compare mean episode length against wall-clock time with and without pixels.